<a href="https://colab.research.google.com/github/evakonstantinova/EfficientNet-B0/blob/main/EfficientNet_B0_Noise_Infused_Callibration_Data_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# DATASET DOWNLOAD AND STRATIFIED SPLIT

from pathlib import Path
from collections import Counter

import kagglehub
from sklearn.model_selection import train_test_split

path = kagglehub.dataset_download(
    "masoudnickparvar/brain-tumor-mri-dataset"
)

classes = ["glioma", "meningioma", "notumor", "pituitary"]

all_files = []
all_labels = []

for class_name in classes:
    for folder in ["Training", "Testing"]:
        class_path = Path(path) / folder / class_name

        for file_path in class_path.iterdir():
            if file_path.is_file():
                all_files.append(str(file_path))
                all_labels.append(class_name)

train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files,
    all_labels,
    test_size=0.30,
    random_state=42,
    stratify=all_labels
)

val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files,
    temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

print("Dataset path:", path)
print("Total images:", len(all_files))
print("Overall distribution:", Counter(all_labels))
print("Training:", len(train_files), Counter(train_labels))
print("Validation:", len(val_files), Counter(val_labels))
print("Testing:", len(test_files), Counter(test_labels))

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Dataset path: /kaggle/input/brain-tumor-mri-dataset
Total images: 7200
Overall distribution: Counter({'glioma': 1800, 'meningioma': 1800, 'notumor': 1800, 'pituitary': 1800})
Training: 5040 Counter({'notumor': 1260, 'pituitary': 1260, 'glioma': 1260, 'meningioma': 1260})
Validation: 1080 Counter({'notumor': 270, 'glioma': 270, 'meningioma': 270, 'pituitary': 270})
Testing: 1080 Counter({'meningioma': 270, 'notumor': 270, 'glioma': 270, 'pituitary': 270})


In [2]:
# DATA PREPROCESSING AND DATALOADERS

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class_to_idx = {
    "glioma": 0,
    "meningioma": 1,
    "notumor": 2,
    "pituitary": 3
}

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class BrainTumorDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image = Image.open(self.files[idx]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_to_idx[self.labels[idx]]
        return image, label

train_dataset = BrainTumorDataset(
    train_files,
    train_labels,
    transform=train_transform
)

val_dataset = BrainTumorDataset(
    val_files,
    val_labels,
    transform=val_test_transform
)

test_dataset = BrainTumorDataset(
    test_files,
    test_labels,
    transform=val_test_transform
)

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 5040
Validation dataset: 1080
Test dataset: 1080


In [3]:
# PENNYLANE DOWNLOAD
!pip -q install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 121.3 MB/s eta 0:00:00


In [4]:
# IMPORTING LIBRARIES AND CHECKING VERSIONS

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision.models import EfficientNet_B0_Weights

import pennylane as qml

print("PyTorch version:", torch.__version__)
print("PennyLane version:", qml.__version__)

PyTorch version: 2.11.0+cu128
PennyLane version: 0.45.1


In [5]:
# DEFINING THE QUANTUM CIRCUIT

# DEFINE THE QUANTUM CIRCUIT

N_QUBITS = 4
N_Q_LAYERS = 2

quantum_device = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

@qml.qnode(
    quantum_device,
    interface="torch",
    diff_method="backprop"
)
def quantum_circuit(inputs, weights):

    # ENCODE 4 CLASSICAL FEATURES INTO 4 QUBITS
    qml.AngleEmbedding(
        inputs,
        wires=range(N_QUBITS),
        rotation="Y"
    )

    # APPLY TRAINABLE QUANTUM LAYERS
    qml.StronglyEntanglingLayers(
        weights,
        wires=range(N_QUBITS)
    )

    # MEASURE EACH QUBIT
    return [
        qml.expval(qml.PauliZ(i))
        for i in range(N_QUBITS)
    ]


weight_shapes = {
    "weights": (
        N_Q_LAYERS,
        N_QUBITS,
        3
    )
}

quantum_layer = qml.qnn.TorchLayer(
    quantum_circuit,
    weight_shapes
)

print("Quantum layer created successfully.")
print("Qubits:", N_QUBITS)
print("Quantum layers:", N_Q_LAYERS)

Quantum layer created successfully.
Qubits: 4
Quantum layers: 2


In [6]:
# BUILD THE EFFICIENTNET-B0 + QUANTUM HYBRID MODEL

# BUILD THE EFFICIENTNET-B0 + QUANTUM HYBRID MODEL

class HQNN(nn.Module):

    def __init__(self, quantum_layer):
        super().__init__()

        # LOAD IMAGENET-PRETRAINED EFFICIENTNET-B0
        efficientnet = models.efficientnet_b0(
            weights=EfficientNet_B0_Weights.DEFAULT
        )

        # KEEP ONLY THE FEATURE EXTRACTOR
        self.features = efficientnet.features
        self.avgpool = efficientnet.avgpool

        # FREEZE EFFICIENTNET FEATURE EXTRACTOR
        for param in self.features.parameters():
            param.requires_grad = False

        # REDUCE 1280 EFFICIENTNET FEATURES TO 4
        self.feature_reduction = nn.Linear(
            1280,
            N_QUBITS
        )

        # QUANTUM LAYER
        self.quantum_layer = quantum_layer

        # FINAL FOUR-CLASS CLASSIFIER
        self.classifier = nn.Linear(
            N_QUBITS,
            4
        )

    def forward(self, x):

        # EXTRACT EFFICIENTNET FEATURES
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        # REDUCE 1280 FEATURES TO 4
        x = self.feature_reduction(x)

        # SCALE VALUES FOR QUANTUM ANGLE ENCODING
        x = torch.tanh(x) * torch.pi

        # MOVE QUANTUM INPUT TO CPU
        x = x.cpu()

        # RUN THE QUANTUM CIRCUIT ON CPU
        x = self.quantum_layer(x)

        # MOVE QUANTUM OUTPUT BACK TO THE CLASSIFIER DEVICE
        classifier_device = next(
            self.classifier.parameters()
        ).device

        x = x.to(classifier_device)

        # PRODUCE FOUR-CLASS OUTPUT
        x = self.classifier(x)

        return x


hqnn_model = HQNN(
    quantum_layer
)

print(hqnn_model)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 47.9MB/s]


HQNN(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivation(
   

In [7]:
# COUNT TOTAL AND TRAINABLE PARAMETERS

total_params = sum(
    p.numel()
    for p in hqnn_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in hqnn_model.parameters()
    if p.requires_grad
)

quantum_params = sum(
    p.numel()
    for p in hqnn_model.quantum_layer.parameters()
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Quantum trainable parameters: {quantum_params:,}")

Total parameters: 4,012,716
Trainable parameters: 5,168
Quantum trainable parameters: 24


In [8]:
# TEST THE HQNN WITH ONE SMALL BATCH

hqnn_model.eval()

images, labels = next(iter(train_loader))

# USE ONLY 2 IMAGES FOR THE TEST
test_images = images[:2]

with torch.no_grad():
    outputs = hqnn_model(test_images)

print("Input shape:", test_images.shape)
print("Output shape:", outputs.shape)
print("Output:")
print(outputs)

Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 4])
Output:
tensor([[ 0.1969, -0.5118, -0.4063, -0.2913],
        [-0.0340, -0.4817, -0.2752, -0.2210]])


In [9]:
# PREPARE THE FROZEN EFFICIENTNET-B0 FEATURE EXTRACTOR

feature_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Feature extraction device:", feature_device)

# MOVE ONLY THE FROZEN EFFICIENTNET PART TO THE AVAILABLE DEVICE
hqnn_model.features = hqnn_model.features.to(feature_device)
hqnn_model.avgpool = hqnn_model.avgpool.to(feature_device)

# KEEP THE FROZEN FEATURE EXTRACTOR IN EVALUATION MODE
hqnn_model.features.eval()
hqnn_model.avgpool.eval()


def extract_efficientnet_features(data_loader):

    all_features = []
    all_labels = []

    # NO GRADIENTS ARE REQUIRED FOR THE FROZEN EFFICIENTNET FEATURE EXTRACTOR
    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(feature_device)

            # EXTRACT EFFICIENTNET-B0 FEATURES
            features = hqnn_model.features(images)
            features = hqnn_model.avgpool(features)
            features = torch.flatten(features, 1)

            # MOVE THE 1280-DIMENSIONAL FEATURES BACK TO CPU
            all_features.append(features.cpu())
            all_labels.append(labels.cpu())

    all_features = torch.cat(all_features, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return all_features, all_labels


print("Feature extraction function created successfully.")

Feature extraction device: cuda
Feature extraction function created successfully.


In [10]:
# CONFIGURE THE NOISE-FREE HQNN TRAINING

# KEEP THE TRAINABLE HYBRID CLASSIFIER ON CPU FOR THE PENNYLANE SIMULATOR
hqnn_model.feature_reduction = hqnn_model.feature_reduction.cpu()
hqnn_model.quantum_layer = hqnn_model.quantum_layer.cpu()
hqnn_model.classifier = hqnn_model.classifier.cpu()

# DEFINE THE LOSS FUNCTION
criterion = nn.CrossEntropyLoss()

# OPTIMIZE ONLY THE TRAINABLE HQNN PARAMETERS
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, hqnn_model.parameters()),
    lr=0.001
)

# SET THE NUMBER OF TRAINING EPOCHS
NUM_EPOCHS = 10

# INITIALIZE BEST VALIDATION LOSS
best_val_loss = float("inf")

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Number of epochs:", NUM_EPOCHS)
print("Noise-free quantum device: default.qubit")

Loss function: CrossEntropyLoss
Optimizer: Adam
Learning rate: 0.001
Number of epochs: 10
Noise-free quantum device: default.qubit


In [11]:
# EXTRACT EFFICIENTNET-B0 FEATURES FROM THE VALIDATION DATASET

print("Extracting validation features...")

val_features, val_labels = extract_efficientnet_features(val_loader)

print("Validation feature extraction completed.")
print("Validation features shape:", val_features.shape)
print("Validation labels shape:", val_labels.shape)

Extracting validation features...
Validation feature extraction completed.
Validation features shape: torch.Size([1080, 1280])
Validation labels shape: torch.Size([1080])


In [12]:
# TRAIN THE NOISE-FREE HQNN

import time
from sklearn.metrics import f1_score
from torch.utils.data import TensorDataset, DataLoader

# STORE TRAINING HISTORY
train_losses = []
train_accuracies = []
train_f1_scores = []

val_losses = []
val_accuracies = []
val_f1_scores = []

best_val_loss = float("inf")
best_epoch = 0

# START TOTAL TRAINING TIMER
training_start_time = time.time()

for epoch in range(NUM_EPOCHS):

    print(f"\nEPOCH {epoch + 1}/{NUM_EPOCHS}")
    print("Extracting augmented training features...")

    # EXTRACT NEW TRAINING FEATURES EACH EPOCH
    # THIS PRESERVES RANDOM TRAINING AUGMENTATION
    train_features, train_labels = extract_efficientnet_features(
        train_loader
    )

    # CREATE A FEATURE-LEVEL TRAINING LOADER
    train_feature_dataset = TensorDataset(
        train_features,
        train_labels
    )

    train_feature_loader = DataLoader(
        train_feature_dataset,
        batch_size=32,
        shuffle=True
    )

    # SET THE TRAINABLE HQNN COMPONENTS TO TRAINING MODE
    hqnn_model.feature_reduction.train()
    hqnn_model.quantum_layer.train()
    hqnn_model.classifier.train()

    running_train_loss = 0.0
    train_predictions = []
    train_targets = []

    # TRAIN THE HYBRID CLASSIFIER
    for features, labels in train_feature_loader:

        optimizer.zero_grad()

        # REDUCE 1280 EFFICIENTNET FEATURES TO 4
        outputs = hqnn_model.feature_reduction(features)

        # SCALE THE FOUR FEATURES FOR QUANTUM ANGLE ENCODING
        outputs = torch.tanh(outputs) * torch.pi

        # PASS THE FEATURES THROUGH THE QUANTUM CIRCUIT
        outputs = hqnn_model.quantum_layer(outputs)

        # PRODUCE FOUR-CLASS OUTPUT
        outputs = hqnn_model.classifier(outputs)

        # CALCULATE CLASSIFICATION LOSS
        loss = criterion(outputs, labels)

        # CALCULATE GRADIENTS AND UPDATE TRAINABLE PARAMETERS
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * features.size(0)

        predictions = torch.argmax(outputs, dim=1)

        train_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        train_targets.extend(
            labels.cpu().numpy()
        )

    # CALCULATE TRAINING METRICS
    epoch_train_loss = (
        running_train_loss / len(train_feature_dataset)
    )

    epoch_train_accuracy = (
        sum(
            p == t
            for p, t in zip(
                train_predictions,
                train_targets
            )
        )
        / len(train_targets)
    )

    epoch_train_f1 = f1_score(
        train_targets,
        train_predictions,
        average="macro"
    )

    # SET THE TRAINABLE HQNN COMPONENTS TO EVALUATION MODE
    hqnn_model.feature_reduction.eval()
    hqnn_model.quantum_layer.eval()
    hqnn_model.classifier.eval()

    # CREATE THE VALIDATION FEATURE LOADER
    val_feature_dataset = TensorDataset(
        val_features,
        val_labels
    )

    val_feature_loader = DataLoader(
        val_feature_dataset,
        batch_size=32,
        shuffle=False
    )

    running_val_loss = 0.0
    val_predictions = []
    val_targets = []

    # EVALUATE ON THE VALIDATION DATASET
    with torch.no_grad():

        for features, labels in val_feature_loader:

            # REDUCE 1280 FEATURES TO 4
            outputs = hqnn_model.feature_reduction(features)

            # SCALE FEATURES FOR QUANTUM ANGLE ENCODING
            outputs = torch.tanh(outputs) * torch.pi

            # PASS THROUGH THE NOISE-FREE QUANTUM CIRCUIT
            outputs = hqnn_model.quantum_layer(outputs)

            # PRODUCE FOUR-CLASS OUTPUT
            outputs = hqnn_model.classifier(outputs)

            loss = criterion(outputs, labels)

            running_val_loss += (
                loss.item() * features.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_predictions.extend(
                predictions.cpu().numpy()
            )

            val_targets.extend(
                labels.cpu().numpy()
            )

    # CALCULATE VALIDATION METRICS
    epoch_val_loss = (
        running_val_loss / len(val_feature_dataset)
    )

    epoch_val_accuracy = (
        sum(
            p == t
            for p, t in zip(
                val_predictions,
                val_targets
            )
        )
        / len(val_targets)
    )

    epoch_val_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro"
    )

    # STORE EPOCH RESULTS
    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_accuracy)
    train_f1_scores.append(epoch_train_f1)

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_accuracy)
    val_f1_scores.append(epoch_val_f1)

    # SAVE THE CHECKPOINT WITH THE LOWEST VALIDATION LOSS
    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss
        best_epoch = epoch + 1

        torch.save(
            hqnn_model.state_dict(),
            "/content/best_hqnn_noisefree.pth"
        )

        checkpoint_message = " <-- BEST CHECKPOINT"

    else:
        checkpoint_message = ""

    # PRINT EPOCH RESULTS
    print(
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_accuracy:.4f} | "
        f"Train Macro F1: {epoch_train_f1:.4f}"
    )

    print(
        f"Val Loss:   {epoch_val_loss:.4f} | "
        f"Val Acc:   {epoch_val_accuracy:.4f} | "
        f"Val Macro F1:   {epoch_val_f1:.4f}"
        f"{checkpoint_message}"
    )


# CALCULATE TOTAL TRAINING WALL TIME
training_end_time = time.time()

hqnn_training_time = (
    training_end_time - training_start_time
)

minutes = int(hqnn_training_time // 60)
seconds = int(hqnn_training_time % 60)

print("\nNOISE-FREE HQNN TRAINING COMPLETED")
print("Best checkpoint epoch:", best_epoch)
print(f"Best validation loss: {best_val_loss:.4f}")
print(
    f"Total training wall time: "
    f"{minutes} min {seconds} s"
)


EPOCH 1/10
Extracting augmented training features...
Train Loss: 1.1978 | Train Acc: 0.5032 | Train Macro F1: 0.4518
Val Loss:   1.0854 | Val Acc:   0.6954 | Val Macro F1:   0.6531 <-- BEST CHECKPOINT

EPOCH 2/10
Extracting augmented training features...
Train Loss: 0.9659 | Train Acc: 0.7262 | Train Macro F1: 0.6879
Val Loss:   0.8810 | Val Acc:   0.7222 | Val Macro F1:   0.6856 <-- BEST CHECKPOINT

EPOCH 3/10
Extracting augmented training features...
Train Loss: 0.7859 | Train Acc: 0.7518 | Train Macro F1: 0.7149
Val Loss:   0.7791 | Val Acc:   0.7315 | Val Macro F1:   0.6740 <-- BEST CHECKPOINT

EPOCH 4/10
Extracting augmented training features...
Train Loss: 0.6626 | Train Acc: 0.8250 | Train Macro F1: 0.8165
Val Loss:   0.6211 | Val Acc:   0.8241 | Val Macro F1:   0.8152 <-- BEST CHECKPOINT

EPOCH 5/10
Extracting augmented training features...
Train Loss: 0.5494 | Train Acc: 0.8720 | Train Macro F1: 0.8705
Val Loss:   0.5639 | Val Acc:   0.8380 | Val Macro F1:   0.8306 <-- BEST C

In [13]:
from google.colab import files

files.download("/content/best_hqnn_noisefree.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
# LOAD THE BEST STAGE 1 HQNN CHECKPOINT

hqnn_model.load_state_dict(
    torch.load(
        "/content/best_hqnn_noisefree.pth",
        weights_only=True
    )
)

print("Best Stage 1 HQNN checkpoint loaded.")


# FREEZE THE COMPLETE EFFICIENTNET-B0 FEATURE EXTRACTOR FIRST

for param in hqnn_model.features.parameters():
    param.requires_grad = False


# UNFREEZE THE FINAL TWO EFFICIENTNET-B0 FEATURE SECTIONS

for param in hqnn_model.features[-2:].parameters():
    param.requires_grad = True


# KEEP THE FEATURE REDUCTION LAYER TRAINABLE

for param in hqnn_model.feature_reduction.parameters():
    param.requires_grad = True


# KEEP THE QUANTUM CIRCUIT TRAINABLE

for param in hqnn_model.quantum_layer.parameters():
    param.requires_grad = True


# KEEP THE FINAL CLASSIFIER TRAINABLE

for param in hqnn_model.classifier.parameters():
    param.requires_grad = True


# COUNT TOTAL AND TRAINABLE PARAMETERS FOR FINE-TUNING

total_params_finetune = sum(
    p.numel()
    for p in hqnn_model.parameters()
)

trainable_params_finetune = sum(
    p.numel()
    for p in hqnn_model.parameters()
    if p.requires_grad
)

quantum_params_finetune = sum(
    p.numel()
    for p in hqnn_model.quantum_layer.parameters()
    if p.requires_grad
)


print(f"Total parameters: {total_params_finetune:,}")
print(
    f"Fine-tuning trainable parameters: "
    f"{trainable_params_finetune:,}"
)
print(
    f"Quantum trainable parameters: "
    f"{quantum_params_finetune:,}"
)

Best Stage 1 HQNN checkpoint loaded.
Total parameters: 4,012,716
Fine-tuning trainable parameters: 1,134,560
Quantum trainable parameters: 24


In [15]:
# CONFIGURE HQNN FINE-TUNING

# SELECT CUDA FOR THE CLASSICAL COMPONENTS
finetune_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# MOVE THE COMPLETE MODEL TO THE SELECTED DEVICE FIRST
hqnn_model = hqnn_model.to(
    finetune_device
)

# MOVE THE QUANTUM LAYER BACK TO CPU
hqnn_model.quantum_layer = (
    hqnn_model.quantum_layer.cpu()
)

# DEFINE THE LOSS FUNCTION
criterion = nn.CrossEntropyLoss()

# SEPARATE CLASSICAL AND QUANTUM TRAINABLE PARAMETERS
classical_trainable_params = [
    param
    for name, param in hqnn_model.named_parameters()
    if param.requires_grad
    and not name.startswith("quantum_layer")
]

quantum_trainable_params = list(
    hqnn_model.quantum_layer.parameters()
)

# CONFIGURE ADAMW FOR MIXED CPU AND GPU PARAMETERS
optimizer = torch.optim.AdamW(
    [
        {
            "params": classical_trainable_params
        },
        {
            "params": quantum_trainable_params
        }
    ],
    lr=0.0001,
    weight_decay=0.0001,
    foreach=False
)

# SET THE MAXIMUM NUMBER OF FINE-TUNING EPOCHS
FINE_TUNE_EPOCHS = 10

# CONFIGURE EARLY STOPPING
PATIENCE = 3
MIN_DELTA = 0.001
EARLY_STOPPING_START_EPOCH = 6

best_finetune_val_loss = float("inf")
best_finetune_epoch = 0
patience_counter = 0

# COUNT TRAINABLE PARAMETERS
trainable_params_finetune = sum(
    p.numel()
    for p in hqnn_model.parameters()
    if p.requires_grad
)

print(
    "Classical fine-tuning device:",
    finetune_device
)

print(
    "Quantum layer device:",
    next(
        hqnn_model.quantum_layer.parameters()
    ).device
)

print("Optimizer: AdamW")
print("Learning rate: 0.0001")
print("Weight decay: 0.0001")
print("Maximum epochs:", FINE_TUNE_EPOCHS)
print("Early stopping patience:", PATIENCE)
print("Early stopping min_delta:", MIN_DELTA)

print(
    "Fine-tuning trainable parameters:",
    f"{trainable_params_finetune:,}"
)

Classical fine-tuning device: cuda
Quantum layer device: cpu
Optimizer: AdamW
Learning rate: 0.0001
Weight decay: 0.0001
Maximum epochs: 10
Early stopping patience: 3
Early stopping min_delta: 0.001
Fine-tuning trainable parameters: 1,134,560


In [17]:
# FINE-TUNE THE NOISE-FREE HQNN END-TO-END

import time
from sklearn.metrics import f1_score

# STORE FINE-TUNING HISTORY
finetune_train_losses = []
finetune_train_accuracies = []
finetune_train_f1_scores = []

finetune_val_losses = []
finetune_val_accuracies = []
finetune_val_f1_scores = []

best_finetune_val_loss = float("inf")
best_finetune_epoch = 0

patience_counter = 0
early_stopping_best_loss = float("inf")

# START THE FINE-TUNING TIMER
finetune_start_time = time.time()

for epoch in range(FINE_TUNE_EPOCHS):

    # SET THE COMPLETE HQNN TO TRAINING MODE
    hqnn_model.train()

    running_train_loss = 0.0
    train_predictions = []
    train_targets = []

    # TRAIN THE HQNN ON THE MRI TRAINING DATASET
    for images, labels in train_loader:

        # MOVE THE BATCH TO THE FINE-TUNING DEVICE
        images = images.to(finetune_device)
        labels = labels.to(finetune_device)

        # RESET GRADIENTS
        optimizer.zero_grad()

        # RUN THE COMPLETE HYBRID MODEL
        outputs = hqnn_model(images)

        # CALCULATE CLASSIFICATION LOSS
        loss = criterion(
            outputs,
            labels
        )

        # CALCULATE GRADIENTS
        loss.backward()

        # UPDATE THE TRAINABLE PARAMETERS
        optimizer.step()

        running_train_loss += (
            loss.item() * images.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        train_targets.extend(
            labels.detach().cpu().numpy()
        )

    # CALCULATE TRAINING METRICS
    epoch_train_loss = (
        running_train_loss
        / len(train_loader.dataset)
    )

    epoch_train_accuracy = (
        sum(
            p == t
            for p, t in zip(
                train_predictions,
                train_targets
            )
        )
        / len(train_targets)
    )

    epoch_train_f1 = f1_score(
        train_targets,
        train_predictions,
        average="macro"
    )

    # SWITCH THE HQNN TO EVALUATION MODE
    hqnn_model.eval()

    running_val_loss = 0.0
    val_predictions = []
    val_targets = []

    # EVALUATE THE HQNN ON THE VALIDATION DATASET
    with torch.no_grad():

        for images, labels in val_loader:

            # MOVE THE BATCH TO THE FINE-TUNING DEVICE
            images = images.to(finetune_device)
            labels = labels.to(finetune_device)

            # RUN THE COMPLETE HYBRID MODEL
            outputs = hqnn_model(images)

            # CALCULATE VALIDATION LOSS
            loss = criterion(
                outputs,
                labels
            )

            running_val_loss += (
                loss.item() * images.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_predictions.extend(
                predictions.detach().cpu().numpy()
            )

            val_targets.extend(
                labels.detach().cpu().numpy()
            )

    # CALCULATE VALIDATION METRICS
    epoch_val_loss = (
        running_val_loss
        / len(val_loader.dataset)
    )

    epoch_val_accuracy = (
        sum(
            p == t
            for p, t in zip(
                val_predictions,
                val_targets
            )
        )
        / len(val_targets)
    )

    epoch_val_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro"
    )

    # STORE THE EPOCH RESULTS
    finetune_train_losses.append(
        epoch_train_loss
    )

    finetune_train_accuracies.append(
        epoch_train_accuracy
    )

    finetune_train_f1_scores.append(
        epoch_train_f1
    )

    finetune_val_losses.append(
        epoch_val_loss
    )

    finetune_val_accuracies.append(
        epoch_val_accuracy
    )

    finetune_val_f1_scores.append(
        epoch_val_f1
    )

    # SAVE THE CHECKPOINT WITH THE LOWEST VALIDATION LOSS
    if epoch_val_loss < best_finetune_val_loss:

        best_finetune_val_loss = epoch_val_loss
        best_finetune_epoch = epoch + 1

        torch.save(
            hqnn_model.state_dict(),
            "/content/best_hqnn_noisefree_finetuned.pth"
        )

        checkpoint_message = " <-- BEST CHECKPOINT"

    else:

        checkpoint_message = ""

    # PRINT THE EPOCH RESULTS
    print(
        f"Epoch {epoch + 1:02d}: "
        f"Train Loss {epoch_train_loss:.4f}, "
        f"Acc {epoch_train_accuracy:.4f}, "
        f"F1 {epoch_train_f1:.4f} | "
        f"Val Loss {epoch_val_loss:.4f}, "
        f"Acc {epoch_val_accuracy:.4f}, "
        f"F1 {epoch_val_f1:.4f}"
        f"{checkpoint_message}"
    )

    # UPDATE EARLY STOPPING AFTER THE FIRST FIVE EPOCHS
    if epoch + 1 >= EARLY_STOPPING_START_EPOCH:

        if (
            epoch_val_loss
            < early_stopping_best_loss - MIN_DELTA
        ):

            early_stopping_best_loss = epoch_val_loss
            patience_counter = 0

        else:

            patience_counter += 1

            print(
                f"Early stopping counter: "
                f"{patience_counter}/{PATIENCE}"
            )

            if patience_counter >= PATIENCE:

                print(
                    f"Early stopping triggered "
                    f"at epoch {epoch + 1}."
                )

                break

    else:

        if epoch_val_loss < early_stopping_best_loss:

            early_stopping_best_loss = epoch_val_loss


# CALCULATE TOTAL FINE-TUNING WALL TIME
finetune_end_time = time.time()

hqnn_finetune_time = (
    finetune_end_time
    - finetune_start_time
)

finetune_minutes = int(
    hqnn_finetune_time // 60
)

finetune_seconds = int(
    hqnn_finetune_time % 60
)

# CALCULATE THE COMPLETE HQNN TRAINING TIME
total_hqnn_training_time = (
    hqnn_training_time
    + hqnn_finetune_time
)

total_minutes = int(
    total_hqnn_training_time // 60
)

total_seconds = int(
    total_hqnn_training_time % 60
)

print("\nNOISE-FREE HQNN FINE-TUNING COMPLETED")

print(
    "Best fine-tuning checkpoint epoch:",
    best_finetune_epoch
)

print(
    f"Best fine-tuning validation loss: "
    f"{best_finetune_val_loss:.4f}"
)

print(
    f"Fine-tuning wall time: "
    f"{finetune_minutes} min "
    f"{finetune_seconds} s"
)

print(
    f"Total HQNN training wall time: "
    f"{total_minutes} min "
    f"{total_seconds} s"
)

Epoch 01: Train Loss 0.2417, Acc 0.9343, F1 0.9342 | Val Loss 0.2247, Acc 0.9426, F1 0.9422 <-- BEST CHECKPOINT
Epoch 02: Train Loss 0.2208, Acc 0.9399, F1 0.9396 | Val Loss 0.2378, Acc 0.9361, F1 0.9358
Epoch 03: Train Loss 0.2124, Acc 0.9450, F1 0.9448 | Val Loss 0.2004, Acc 0.9472, F1 0.9469 <-- BEST CHECKPOINT
Epoch 04: Train Loss 0.2019, Acc 0.9476, F1 0.9474 | Val Loss 0.1963, Acc 0.9444, F1 0.9442 <-- BEST CHECKPOINT
Epoch 05: Train Loss 0.1849, Acc 0.9556, F1 0.9555 | Val Loss 0.1828, Acc 0.9500, F1 0.9497 <-- BEST CHECKPOINT
Epoch 06: Train Loss 0.1781, Acc 0.9560, F1 0.9559 | Val Loss 0.1736, Acc 0.9537, F1 0.9534 <-- BEST CHECKPOINT
Epoch 07: Train Loss 0.1636, Acc 0.9583, F1 0.9583 | Val Loss 0.1715, Acc 0.9509, F1 0.9506 <-- BEST CHECKPOINT
Epoch 08: Train Loss 0.1550, Acc 0.9629, F1 0.9628 | Val Loss 0.1598, Acc 0.9565, F1 0.9563 <-- BEST CHECKPOINT
Epoch 09: Train Loss 0.1527, Acc 0.9621, F1 0.9620 | Val Loss 0.1552, Acc 0.9593, F1 0.9590 <-- BEST CHECKPOINT
Epoch 10: Tr

In [18]:
from google.colab import files

files.download("/content/best_hqnn_noisefree_finetuned.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
# FINAL TEST OF THE BEST FINE-TUNED NOISE-FREE HQNN

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# LOAD THE BEST FINE-TUNED HQNN CHECKPOINT

best_hqnn_state = torch.load(
    "/content/best_hqnn_noisefree_finetuned.pth",
    map_location="cpu",
    weights_only=True
)

hqnn_model.load_state_dict(
    best_hqnn_state
)

print("Best fine-tuned HQNN checkpoint loaded.")


# RESTORE THE MIXED CPU-GPU DEVICE CONFIGURATION

hqnn_model.features = (
    hqnn_model.features.to(finetune_device)
)

hqnn_model.avgpool = (
    hqnn_model.avgpool.to(finetune_device)
)

hqnn_model.feature_reduction = (
    hqnn_model.feature_reduction.to(finetune_device)
)

hqnn_model.classifier = (
    hqnn_model.classifier.to(finetune_device)
)

hqnn_model.quantum_layer = (
    hqnn_model.quantum_layer.cpu()
)


# SET THE COMPLETE MODEL TO EVALUATION MODE

hqnn_model.eval()


# STORE FINAL TEST OUTPUTS

all_test_targets = []
all_test_predictions = []
all_test_probabilities = []


# EVALUATE THE MODEL ON THE COMPLETE TEST DATASET

with torch.no_grad():

    for images, labels in test_loader:

        # MOVE MRI IMAGES AND LABELS TO THE CLASSICAL DEVICE

        images = images.to(
            finetune_device
        )

        labels = labels.to(
            finetune_device
        )

        # RUN THE COMPLETE FINE-TUNED HQNN

        outputs = hqnn_model(
            images
        )

        # CONVERT LOGITS TO CLASS PROBABILITIES

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        # SELECT THE CLASS WITH THE HIGHEST LOGIT

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        # STORE TARGET LABELS

        all_test_targets.extend(
            labels.cpu().numpy()
        )

        # STORE PREDICTED LABELS

        all_test_predictions.extend(
            predictions.cpu().numpy()
        )

        # STORE CLASS PROBABILITIES

        all_test_probabilities.extend(
            probabilities.cpu().numpy()
        )


# CONVERT RESULTS TO NUMPY ARRAYS

all_test_targets = np.array(
    all_test_targets
)

all_test_predictions = np.array(
    all_test_predictions
)

all_test_probabilities = np.array(
    all_test_probabilities
)


# CALCULATE FINAL TEST METRICS

test_accuracy = accuracy_score(
    all_test_targets,
    all_test_predictions
)

test_precision = precision_score(
    all_test_targets,
    all_test_predictions,
    average="macro"
)

test_recall = recall_score(
    all_test_targets,
    all_test_predictions,
    average="macro"
)

test_f1 = f1_score(
    all_test_targets,
    all_test_predictions,
    average="macro"
)

test_roc_auc = roc_auc_score(
    all_test_targets,
    all_test_probabilities,
    multi_class="ovr",
    average="macro"
)


# PRINT FINAL TEST RESULTS

print(
    "\nFINAL FINE-TUNED NOISE-FREE HQNN TEST RESULTS"
)

print(
    f"Accuracy:        {test_accuracy:.4f}"
)

print(
    f"Macro Precision: {test_precision:.4f}"
)

print(
    f"Macro Recall:    {test_recall:.4f}"
)

print(
    f"Macro F1-score:  {test_f1:.4f}"
)

print(
    f"Macro ROC-AUC:   {test_roc_auc:.4f}"
)


# PRINT CLASSIFICATION RESULTS FOR EACH TUMOR CLASS

print(
    "\nCLASSIFICATION REPORT"
)

print(
    classification_report(
        all_test_targets,
        all_test_predictions,
        target_names=[
            "Glioma",
            "Meningioma",
            "No Tumor",
            "Pituitary"
        ],
        digits=4
    )
)

Best fine-tuned HQNN checkpoint loaded.

FINAL FINE-TUNED NOISE-FREE HQNN TEST RESULTS
Accuracy:        0.9435
Macro Precision: 0.9431
Macro Recall:    0.9435
Macro F1-score:  0.9431
Macro ROC-AUC:   0.9919

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Glioma     0.9302    0.8889    0.9091       270
  Meningioma     0.9037    0.9037    0.9037       270
    No Tumor     0.9745    0.9926    0.9835       270
   Pituitary     0.9639    0.9889    0.9762       270

    accuracy                         0.9435      1080
   macro avg     0.9431    0.9435    0.9431      1080
weighted avg     0.9431    0.9435    0.9431      1080



In [20]:
# INSTALL COMPATIBLE QUANTUM SOFTWARE VERSIONS

!pip uninstall -y qiskit qiskit-aer qiskit-ibm-runtime pennylane-qiskit > /dev/null 2>&1

!pip install -q \
    "qiskit==2.3.0" \
    "qiskit-aer==0.17.2" \
    "qiskit-ibm-runtime==0.45.1" \
    "pennylane-qiskit==0.45.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 11.4 MB/s eta 0:00:00


In [21]:
# INSTALL THE MISSING IPYTHON DEPENDENCY

!pip install -q jedi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 35.3 MB/s eta 0:00:00


In [22]:
# CHECK FOR PACKAGE DEPENDENCY CONFLICTS

!pip check

No broken requirements found.


In [23]:
# CONNECT TO IBM QUANTUM PLATFORM

from getpass import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

# ENTER THE IBM QUANTUM API KEY SECURELY
ibm_api_key = getpass(
    "Enter IBM Quantum API key: "
)

# CONNECT TO IBM QUANTUM PLATFORM
service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=ibm_api_key
)

print(
    "Connected to IBM Quantum successfully."
)

Enter IBM Quantum API key: ··········


qiskit_runtime_service._discover_account:WARNING:2026-08-20 08:00:36,270: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-08-20 08:00:40,121: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().


Connected to IBM Quantum successfully.


In [24]:
# LIST AVAILABLE IBM QUANTUM HARDWARE BACKENDS

available_backends = service.backends(
    simulator=False,
    operational=True,
    min_num_qubits=4
)

print("AVAILABLE IBM QUANTUM BACKENDS")
print("--------------------------------")

for backend in available_backends:

    status = backend.status()

    print(f"Backend: {backend.name}")
    print(f"Qubits: {backend.num_qubits}")
    print(f"Pending jobs: {status.pending_jobs}")
    print(f"Operational: {status.operational}")
    print("--------------------------------")

qiskit_runtime_service.backends:WARNING:2026-08-20 08:00:43,778: Loading instance: open-instance, plan: open


AVAILABLE IBM QUANTUM BACKENDS
--------------------------------
Backend: ibm_fez
Qubits: 156
Pending jobs: 2
Operational: True
--------------------------------
Backend: ibm_marrakesh
Qubits: 156
Pending jobs: 0
Operational: True
--------------------------------
Backend: ibm_kingston
Qubits: 156
Pending jobs: 7
Operational: True
--------------------------------


In [25]:
# COMPARE CALIBRATION QUALITY OF THE AVAILABLE IBM QPUS

import numpy as np

backend_names = [
    "ibm_fez",
    "ibm_marrakesh",
    "ibm_kingston"
]

print("IBM QPU CALIBRATION COMPARISON")
print("------------------------------------------------------------")

for backend_name in backend_names:

    # LOAD THE BACKEND
    backend = service.backend(
        backend_name
    )

    # REQUEST THE CURRENT CALIBRATION PROPERTIES
    properties = backend.properties(
        refresh=True
    )

    # STORE QUBIT-LEVEL CALIBRATION VALUES
    t1_values = []
    t2_values = []
    readout_errors = []

    for qubit in range(
        backend.num_qubits
    ):

        try:
            t1 = properties.t1(
                qubit
            )

            if t1 is not None:
                t1_values.append(
                    t1
                )

        except Exception:
            pass

        try:
            t2 = properties.t2(
                qubit
            )

            if t2 is not None:
                t2_values.append(
                    t2
                )

        except Exception:
            pass

        try:
            readout_error = properties.readout_error(
                qubit
            )

            if readout_error is not None:
                readout_errors.append(
                    readout_error
                )

        except Exception:
            pass


    # STORE TWO-QUBIT GATE ERRORS
    two_qubit_errors = []

    for instruction_name in backend.target.keys():

        instruction_properties = (
            backend.target[
                instruction_name
            ]
        )

        # SKIP INSTRUCTIONS WITHOUT A PROPERTY MAP
        if instruction_properties is None:
            continue

        for qubits, instruction_data in (
            instruction_properties.items()
        ):

            # SKIP GLOBAL OPERATIONS WITHOUT SPECIFIC QUBITS
            if qubits is None:
                continue

            # KEEP ONLY TWO-QUBIT OPERATIONS
            if len(qubits) != 2:
                continue

            # SKIP OPERATIONS WITHOUT CALIBRATION ERROR DATA
            if instruction_data is None:
                continue

            if instruction_data.error is None:
                continue

            two_qubit_errors.append(
                instruction_data.error
            )


    # PRINT BACKEND CALIBRATION SUMMARY

    print(
        f"Backend: {backend_name}"
    )

    print(
        f"Calibration timestamp: "
        f"{properties.last_update_date}"
    )

    if t1_values:

        print(
            f"Median T1: "
            f"{np.median(t1_values) * 1e6:.2f} us"
        )

    if t2_values:

        print(
            f"Median T2: "
            f"{np.median(t2_values) * 1e6:.2f} us"
        )

    if readout_errors:

        print(
            f"Median readout error: "
            f"{np.median(readout_errors) * 100:.3f}%"
        )

    if two_qubit_errors:

        print(
            f"Median 2-qubit gate error: "
            f"{np.median(two_qubit_errors) * 100:.3f}%"
        )

        print(
            f"Best 2-qubit gate error: "
            f"{np.min(two_qubit_errors) * 100:.3f}%"
        )

        print(
            f"Worst 2-qubit gate error: "
            f"{np.max(two_qubit_errors) * 100:.3f}%"
        )

    print(
        "------------------------------------------------------------"
    )

qiskit_runtime_service.backends:WARNING:2026-08-20 08:00:54,506: Using instance: open-instance, plan: open


IBM QPU CALIBRATION COMPARISON
------------------------------------------------------------


qiskit_runtime_service.backends:WARNING:2026-08-20 08:00:55,254: Using instance: open-instance, plan: open


Backend: ibm_fez
Calibration timestamp: 2026-08-20 07:48:58+00:00
Median T1: 131.25 us
Median T2: 99.10 us
Median readout error: 0.842%
Median 2-qubit gate error: 0.264%
Best 2-qubit gate error: 0.150%
Worst 2-qubit gate error: 100.000%
------------------------------------------------------------


qiskit_runtime_service.backends:WARNING:2026-08-20 08:00:55,764: Using instance: open-instance, plan: open


Backend: ibm_marrakesh
Calibration timestamp: 2026-08-20 07:36:39+00:00
Median T1: 176.28 us
Median T2: 74.57 us
Median readout error: 1.093%
Median 2-qubit gate error: 0.306%
Best 2-qubit gate error: 0.107%
Worst 2-qubit gate error: 100.000%
------------------------------------------------------------
Backend: ibm_kingston
Calibration timestamp: 2026-08-20 07:25:13+00:00
Median T1: 235.42 us
Median T2: 122.26 us
Median readout error: 0.916%
Median 2-qubit gate error: 0.503%
Best 2-qubit gate error: 0.181%
Worst 2-qubit gate error: 100.000%
------------------------------------------------------------


In [26]:
# SELECT IBM FEZ FOR THE NOISY-SIMULATION EXPERIMENT

IBM_BACKEND_NAME = "ibm_fez"

ibm_backend = service.backend(
    IBM_BACKEND_NAME
)

# REFRESH THE BACKEND TARGET AND CALIBRATION INFORMATION

ibm_backend.refresh()

ibm_properties = ibm_backend.properties(
    refresh=True
)

print(
    "Selected backend:",
    IBM_BACKEND_NAME
)

print(
    "Physical qubits:",
    ibm_backend.num_qubits
)

print(
    "Calibration timestamp:",
    ibm_properties.last_update_date
)

print(
    "IBM FEZ calibration snapshot loaded successfully."
)

qiskit_runtime_service.backends:WARNING:2026-08-20 08:01:02,609: Using instance: open-instance, plan: open


Selected backend: ibm_fez
Physical qubits: 156
Calibration timestamp: 2026-08-20 07:48:58+00:00
IBM FEZ calibration snapshot loaded successfully.


In [27]:
# FIND LOW-ERROR CONNECTED FOUR-QUBIT REGIONS ON IBM FEZ

import numpy as np


# COLLECT CALIBRATED TWO-QUBIT CONNECTIONS

edge_errors = {}

for instruction_name in ibm_backend.target.keys():

    instruction_map = (
        ibm_backend.target[
            instruction_name
        ]
    )

    if instruction_map is None:
        continue

    for qargs, instruction_properties in (
        instruction_map.items()
    ):

        # SKIP GLOBAL OPERATIONS

        if qargs is None:
            continue

        # KEEP ONLY TWO-QUBIT OPERATIONS

        if len(qargs) != 2:
            continue

        # SKIP OPERATIONS WITHOUT ERROR DATA

        if instruction_properties is None:
            continue

        if instruction_properties.error is None:
            continue

        error = float(
            instruction_properties.error
        )

        # EXCLUDE COMPLETELY UNUSABLE CONNECTIONS

        if error >= 1.0:
            continue

        # STORE THE CONNECTION AS AN UNDIRECTED EDGE

        edge = tuple(
            sorted(qargs)
        )

        # KEEP THE LOWEST AVAILABLE ERROR FOR THE CONNECTION

        if (
            edge not in edge_errors
            or error < edge_errors[edge]
        ):

            edge_errors[edge] = error


# BUILD THE PHYSICAL CONNECTIVITY GRAPH

adjacency = {
    qubit: set()
    for qubit in range(
        ibm_backend.num_qubits
    )
}

for qubit_a, qubit_b in edge_errors:

    adjacency[
        qubit_a
    ].add(
        qubit_b
    )

    adjacency[
        qubit_b
    ].add(
        qubit_a
    )


# FIND CONNECTED GROUPS OF FOUR PHYSICAL QUBITS

connected_groups = set()


def expand_group(
    current_group
):

    if len(
        current_group
    ) == 4:

        connected_groups.add(
            tuple(
                sorted(
                    current_group
                )
            )
        )

        return

    neighbours = set()

    for qubit in current_group:

        neighbours.update(
            adjacency[
                qubit
            ]
        )

    neighbours -= (
        current_group
    )

    for neighbour in neighbours:

        expand_group(
            current_group
            | {neighbour}
        )


for start_qubit in range(
    ibm_backend.num_qubits
):

    expand_group(
        {start_qubit}
    )


# CALCULATE A LOW-ERROR SPANNING TREE FOR EACH REGION

def calculate_mst(
    group
):

    group = set(
        group
    )

    internal_edges = []

    for edge, error in (
        edge_errors.items()
    ):

        if (
            edge[0] in group
            and edge[1] in group
        ):

            internal_edges.append(
                (
                    error,
                    edge[0],
                    edge[1]
                )
            )

    internal_edges.sort()

    parent = {
        qubit: qubit
        for qubit in group
    }


    def find(
        qubit
    ):

        while (
            parent[
                qubit
            ] != qubit
        ):

            parent[
                qubit
            ] = parent[
                parent[
                    qubit
                ]
            ]

            qubit = parent[
                qubit
            ]

        return qubit


    selected_edges = []

    for (
        error,
        qubit_a,
        qubit_b
    ) in internal_edges:

        root_a = find(
            qubit_a
        )

        root_b = find(
            qubit_b
        )

        if root_a != root_b:

            parent[
                root_b
            ] = root_a

            selected_edges.append(
                (
                    qubit_a,
                    qubit_b,
                    error
                )
            )

        if len(
            selected_edges
        ) == 3:

            break

    return selected_edges


# SCORE EACH CONNECTED FOUR-QUBIT REGION

candidate_regions = []

for group in connected_groups:

    mst_edges = calculate_mst(
        group
    )

    if len(
        mst_edges
    ) != 3:
        continue

    two_qubit_errors = [
        edge[2]
        for edge in mst_edges
    ]

    readout_errors = []
    t1_values = []
    t2_values = []

    valid_group = True

    for qubit in group:

        try:

            readout_errors.append(
                ibm_properties.readout_error(
                    qubit
                )
            )

            t1_values.append(
                ibm_properties.t1(
                    qubit
                )
            )

            t2_values.append(
                ibm_properties.t2(
                    qubit
                )
            )

        except Exception:

            valid_group = False
            break

    if not valid_group:
        continue

    candidate_regions.append(
        {
            "qubits": group,

            "mean_2q_error": np.mean(
                two_qubit_errors
            ),

            "max_2q_error": np.max(
                two_qubit_errors
            ),

            "mean_readout_error": np.mean(
                readout_errors
            ),

            "median_t1": np.median(
                t1_values
            ),

            "median_t2": np.median(
                t2_values
            ),

            "mst_edges": mst_edges
        }
    )


# RANK REGIONS BY TWO-QUBIT ERROR FIRST

candidate_regions.sort(
    key=lambda region: (
        region[
            "mean_2q_error"
        ],
        region[
            "mean_readout_error"
        ]
    )
)


# PRINT THE TEN BEST REGIONS

print(
    "TOP 10 CONNECTED FOUR-QUBIT REGIONS ON IBM FEZ"
)

print(
    "------------------------------------------------------------"
)

for rank, region in enumerate(
    candidate_regions[:10],
    start=1
):

    print(
        f"Rank {rank}"
    )

    print(
        "Physical qubits:",
        region[
            "qubits"
        ]
    )

    print(
        f"Mean 2-qubit error: "
        f"{region['mean_2q_error'] * 100:.3f}%"
    )

    print(
        f"Maximum 2-qubit error: "
        f"{region['max_2q_error'] * 100:.3f}%"
    )

    print(
        f"Mean readout error: "
        f"{region['mean_readout_error'] * 100:.3f}%"
    )

    print(
        f"Median T1: "
        f"{region['median_t1'] * 1e6:.2f} us"
    )

    print(
        f"Median T2: "
        f"{region['median_t2'] * 1e6:.2f} us"
    )

    print(
        "Connectivity edges:",
        region[
            "mst_edges"
        ]
    )

    print(
        "------------------------------------------------------------"
    )

TOP 10 CONNECTED FOUR-QUBIT REGIONS ON IBM FEZ
------------------------------------------------------------
Rank 1
Physical qubits: (130, 131, 132, 138)
Mean 2-qubit error: 0.177%
Maximum 2-qubit error: 0.197%
Mean readout error: 2.609%
Median T1: 105.11 us
Median T2: 130.42 us
Connectivity edges: [(131, 132, 0.0015166991218628079), (130, 131, 0.0018177961495732808), (131, 138, 0.0019655336496399445)]
------------------------------------------------------------
Rank 2
Physical qubits: (131, 132, 138, 151)
Mean 2-qubit error: 0.179%
Maximum 2-qubit error: 0.197%
Mean readout error: 2.704%
Median T1: 93.61 us
Median T2: 121.39 us
Connectivity edges: [(131, 132, 0.0015166991218628079), (138, 151, 0.0018734293957632098), (131, 138, 0.0019655336496399445)]
------------------------------------------------------------
Rank 3
Physical qubits: (130, 131, 138, 151)
Mean 2-qubit error: 0.189%
Maximum 2-qubit error: 0.197%
Mean readout error: 2.634%
Median T1: 116.04 us
Median T2: 148.50 us
Connec

In [28]:
# TRANSPILE THE FOUR-QUBIT HQNN CIRCUIT ON THE TOP CANDIDATE REGIONS

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.transpiler import generate_preset_pass_manager


# CREATE SYMBOLIC INPUT PARAMETERS

input_parameters = ParameterVector(
    "x",
    4
)

weight_parameters = ParameterVector(
    "w",
    24
)


# BUILD A QISKIT CIRCUIT WITH THE SAME QUANTUM TOPOLOGY AS THE HQNN

reference_circuit = QuantumCircuit(
    4
)


# APPLY ANGLE EMBEDDING WITH Y ROTATIONS

for qubit in range(
    4
):

    reference_circuit.ry(
        input_parameters[
            qubit
        ],
        qubit
    )


# APPLY THE FIRST STRONGLY ENTANGLING LAYER

parameter_index = 0

for qubit in range(
    4
):

    reference_circuit.rz(
        weight_parameters[
            parameter_index
        ],
        qubit
    )

    parameter_index += 1

    reference_circuit.ry(
        weight_parameters[
            parameter_index
        ],
        qubit
    )

    parameter_index += 1

    reference_circuit.rz(
        weight_parameters[
            parameter_index
        ],
        qubit
    )

    parameter_index += 1


# FIRST LAYER USES RANGE 1

for control in range(
    4
):

    target = (
        control + 1
    ) % 4

    reference_circuit.cx(
        control,
        target
    )


# APPLY THE SECOND STRONGLY ENTANGLING LAYER

for qubit in range(
    4
):

    reference_circuit.rz(
        weight_parameters[
            parameter_index
        ],
        qubit
    )

    parameter_index += 1

    reference_circuit.ry(
        weight_parameters[
            parameter_index
        ],
        qubit
    )

    parameter_index += 1

    reference_circuit.rz(
        weight_parameters[
            parameter_index
        ],
        qubit
    )

    parameter_index += 1


# SECOND LAYER USES RANGE 2

for control in range(
    4
):

    target = (
        control + 2
    ) % 4

    reference_circuit.cx(
        control,
        target
    )


print(
    "Reference HQNN quantum circuit created."
)

print(
    "Logical circuit depth:",
    reference_circuit.depth()
)

print(
    "Logical operation counts:",
    reference_circuit.count_ops()
)


# TRANSPILE THE CIRCUIT ON EACH OF THE TOP TEN REGIONS

transpilation_results = []

for rank, region in enumerate(
    candidate_regions[:10],
    start=1
):

    physical_qubits = list(
        region[
            "qubits"
        ]
    )

    # CREATE A BACKEND-AWARE PASS MANAGER

    pass_manager = generate_preset_pass_manager(
        backend=ibm_backend,
        optimization_level=3,
        initial_layout=physical_qubits,
        seed_transpiler=42
    )

    # TRANSPILE THE REFERENCE CIRCUIT

    transpiled_circuit = pass_manager.run(
        reference_circuit
    )


    # COUNT TWO-QUBIT OPERATIONS AFTER TRANSPILATION

    two_qubit_operation_count = 0

    for instruction in (
        transpiled_circuit.data
    ):

        if len(
            instruction.qubits
        ) == 2:

            two_qubit_operation_count += 1


    # GET THE FINAL PHYSICAL LAYOUT

    final_layout = None

    if (
        transpiled_circuit.layout
        is not None
    ):

        final_layout = (
            transpiled_circuit.layout.final_index_layout(
                filter_ancillas=True
            )
        )


    # STORE RESULTS

    transpilation_results.append(
        {
            "rank": rank,
            "qubits": tuple(
                physical_qubits
            ),
            "depth": transpiled_circuit.depth(),
            "two_qubit_ops": two_qubit_operation_count,
            "total_ops": sum(
                transpiled_circuit.count_ops().values()
            ),
            "mean_2q_error": region[
                "mean_2q_error"
            ],
            "max_2q_error": region[
                "max_2q_error"
            ],
            "readout_error": region[
                "mean_readout_error"
            ],
            "final_layout": final_layout,
            "circuit": transpiled_circuit
        }
    )


# RANK PRIMARILY BY TWO-QUBIT OPERATION COUNT,
# THEN CIRCUIT DEPTH,
# THEN CALIBRATION ERROR

transpilation_results.sort(
    key=lambda result: (
        result[
            "two_qubit_ops"
        ],
        result[
            "depth"
        ],
        result[
            "mean_2q_error"
        ],
        result[
            "readout_error"
        ]
    )
)


# PRINT THE RESULTS

print(
    "\nTRANSPILED FOUR-QUBIT REGION COMPARISON"
)

print(
    "------------------------------------------------------------"
)

for result in (
    transpilation_results
):

    print(
        f"Original calibration rank: "
        f"{result['rank']}"
    )

    print(
        f"Physical qubits: "
        f"{result['qubits']}"
    )

    print(
        f"Transpiled circuit depth: "
        f"{result['depth']}"
    )

    print(
        f"Two-qubit operations: "
        f"{result['two_qubit_ops']}"
    )

    print(
        f"Total operations: "
        f"{result['total_ops']}"
    )

    print(
        f"Mean 2-qubit error: "
        f"{result['mean_2q_error'] * 100:.3f}%"
    )

    print(
        f"Maximum 2-qubit error: "
        f"{result['max_2q_error'] * 100:.3f}%"
    )

    print(
        f"Mean readout error: "
        f"{result['readout_error'] * 100:.3f}%"
    )

    print(
        f"Final physical layout: "
        f"{result['final_layout']}"
    )

    print(
        "------------------------------------------------------------"
    )

Reference HQNN quantum circuit created.
Logical circuit depth: 13
Logical operation counts: OrderedDict({'rz': 16, 'ry': 12, 'cx': 8})

TRANSPILED FOUR-QUBIT REGION COMPARISON
------------------------------------------------------------
Original calibration rank: 1
Physical qubits: (130, 131, 132, 138)
Transpiled circuit depth: 61
Two-qubit operations: 14
Total operations: 125
Mean 2-qubit error: 0.177%
Maximum 2-qubit error: 0.197%
Mean readout error: 2.609%
Final physical layout: [131, 138, 132, 130]
------------------------------------------------------------
Original calibration rank: 2
Physical qubits: (131, 132, 138, 151)
Transpiled circuit depth: 53
Two-qubit operations: 17
Total operations: 136
Mean 2-qubit error: 0.179%
Maximum 2-qubit error: 0.197%
Mean readout error: 2.704%
Final physical layout: [151, 132, 138, 131]
------------------------------------------------------------
Original calibration rank: 4
Physical qubits: (91, 92, 93, 98)
Transpiled circuit depth: 54
Two-qub

In [29]:
# FREEZE THE FINAL IBM FEZ FOUR-QUBIT REGION

import json

SELECTED_CALIBRATION_RANK = 7

SELECTED_PHYSICAL_REGION = (
    2,
    3,
    4,
    16
)

LOGICAL_TO_PHYSICAL = [
    3,
    16,
    4,
    2
]

EVALUATION_SHOTS = 1024


# FIND THE SAVED TRANSPILATION RESULT FOR RANK 7

selected_transpilation = next(
    result
    for result in transpilation_results
    if result["rank"] == SELECTED_CALIBRATION_RANK
)


# VERIFY THE SELECTED RESULT

print(
    "FINAL IBM QPU SELECTION"
)

print(
    "Backend:",
    IBM_BACKEND_NAME
)

print(
    "Physical region:",
    SELECTED_PHYSICAL_REGION
)

print(
    "Logical-to-physical mapping:",
    LOGICAL_TO_PHYSICAL
)

print(
    "Transpiled circuit depth:",
    selected_transpilation["depth"]
)

print(
    "Two-qubit operations:",
    selected_transpilation["two_qubit_ops"]
)

print(
    f"Mean 2-qubit error: "
    f"{selected_transpilation['mean_2q_error'] * 100:.3f}%"
)

print(
    f"Maximum 2-qubit error: "
    f"{selected_transpilation['max_2q_error'] * 100:.3f}%"
)

print(
    f"Mean readout error: "
    f"{selected_transpilation['readout_error'] * 100:.3f}%"
)

print(
    "Calibration timestamp:",
    ibm_properties.last_update_date
)

print(
    "Evaluation shots:",
    EVALUATION_SHOTS
)


# SAVE THE COMPLETE IBM CALIBRATION SNAPSHOT

with open(
    "/content/ibm_fez_calibration_snapshot.json",
    "w"
) as file:

    json.dump(
        ibm_properties.to_dict(),
        file,
        indent=2,
        default=str
    )

print(
    "\nCalibration snapshot saved successfully."
)

FINAL IBM QPU SELECTION
Backend: ibm_fez
Physical region: (2, 3, 4, 16)
Logical-to-physical mapping: [3, 16, 4, 2]
Transpiled circuit depth: 69
Two-qubit operations: 20
Mean 2-qubit error: 0.197%
Maximum 2-qubit error: 0.206%
Mean readout error: 2.713%
Calibration timestamp: 2026-08-20 07:48:58+00:00
Evaluation shots: 1024

Calibration snapshot saved successfully.


In [30]:
# BUILD THE IBM FEZ CALIBRATION-BASED NOISE SIMULATOR

from qiskit_aer import AerSimulator


# CREATE A NOISY SIMULATOR FROM THE IBM FEZ CALIBRATION DATA

noisy_ibm_simulator = AerSimulator.from_backend(
    ibm_backend,
    method="density_matrix",
    enable_truncation=True
)


# CREATE AN IDEAL FINITE-SHOT SIMULATOR FOR FAIR COMPARISON

ideal_shot_simulator = AerSimulator(
    method="density_matrix",
    enable_truncation=True
)


print(
    "IBM calibration-based noisy simulator created."
)

print(
    "Backend source:",
    IBM_BACKEND_NAME
)

print(
    "Simulation method: density_matrix"
)

print(
    "Evaluation shots:",
    EVALUATION_SHOTS
)

print(
    "Calibration timestamp:",
    ibm_properties.last_update_date
)

print(
    "Active physical region:",
    SELECTED_PHYSICAL_REGION
)

IBM calibration-based noisy simulator created.
Backend source: ibm_fez
Simulation method: density_matrix
Evaluation shots: 1024
Calibration timestamp: 2026-08-20 07:48:58+00:00
Active physical region: (2, 3, 4, 16)


In [31]:
# TEST THE IDEAL AND IBM CALIBRATION-BASED SIMULATORS

import numpy as np

from qiskit import ClassicalRegister


# USE THE SELECTED TRANSPILED FOUR-QUBIT CIRCUIT

test_circuit = selected_transpilation[
    "circuit"
].copy()


# SET ALL TEST PARAMETERS TO ZERO

zero_parameter_values = {
    parameter: 0.0
    for parameter in test_circuit.parameters
}

test_circuit = test_circuit.assign_parameters(
    zero_parameter_values,
    inplace=False
)


# ADD FOUR CLASSICAL MEASUREMENT BITS

measurement_register = ClassicalRegister(
    4,
    "measurement"
)

test_circuit.add_register(
    measurement_register
)


# MEASURE THE FINAL PHYSICAL LOCATION OF EACH LOGICAL QUBIT

for logical_qubit, physical_qubit in enumerate(
    LOGICAL_TO_PHYSICAL
):

    test_circuit.measure(
        physical_qubit,
        measurement_register[
            logical_qubit
        ]
    )


# RUN THE IDEAL FINITE-SHOT SIMULATION

ideal_job = ideal_shot_simulator.run(
    test_circuit,
    shots=EVALUATION_SHOTS,
    seed_simulator=42
)

ideal_result = ideal_job.result()

ideal_counts = ideal_result.get_counts()


# RUN THE IBM CALIBRATION-BASED NOISY SIMULATION

noisy_job = noisy_ibm_simulator.run(
    test_circuit,
    shots=EVALUATION_SHOTS,
    seed_simulator=42
)

noisy_result = noisy_job.result()

noisy_counts = noisy_result.get_counts()


# CONVERT COUNTS INTO PAULI-Z EXPECTATION VALUES

def counts_to_z_expectations(
    counts,
    number_of_qubits=4
):

    total_shots = sum(
        counts.values()
    )

    expectations = np.zeros(
        number_of_qubits
    )

    for bitstring, count in counts.items():

        clean_bitstring = (
            bitstring.replace(
                " ",
                ""
            )
        )

        bits = clean_bitstring[::-1]

        for qubit in range(
            number_of_qubits
        ):

            if bits[qubit] == "0":

                expectations[
                    qubit
                ] += count

            else:

                expectations[
                    qubit
                ] -= count

    expectations /= total_shots

    return expectations


ideal_expectations = (
    counts_to_z_expectations(
        ideal_counts
    )
)

noisy_expectations = (
    counts_to_z_expectations(
        noisy_counts
    )
)


# PRINT THE TEST RESULTS

print(
    "IDEAL FINITE-SHOT COUNTS:"
)

print(
    ideal_counts
)

print(
    "\nIBM CALIBRATION-NOISY COUNTS:"
)

print(
    noisy_counts
)

print(
    "\nIDEAL PAULI-Z EXPECTATION VALUES:"
)

print(
    np.round(
        ideal_expectations,
        4
    )
)

print(
    "\nNOISY PAULI-Z EXPECTATION VALUES:"
)

print(
    np.round(
        noisy_expectations,
        4
    )
)

print(
    "\nABSOLUTE EXPECTATION-VALUE DIFFERENCE:"
)

print(
    np.round(
        np.abs(
            ideal_expectations
            - noisy_expectations
        ),
        4
    )
)

IDEAL FINITE-SHOT COUNTS:
{'0000': 1024}

IBM CALIBRATION-NOISY COUNTS:
{'1000': 4, '0100': 4, '0001': 20, '0010': 16, '0000': 980}

IDEAL PAULI-Z EXPECTATION VALUES:
[1. 1. 1. 1.]

NOISY PAULI-Z EXPECTATION VALUES:
[0.9609 0.9688 0.9922 0.9922]

ABSOLUTE EXPECTATION-VALUE DIFFERENCE:
[0.0391 0.0312 0.0078 0.0078]


In [32]:
# VERIFY THAT THE QISKIT CIRCUIT MATCHES THE TRAINED PENNYLANE CIRCUIT

import os
import numpy as np
import torch
import pennylane as qml

from qiskit.quantum_info import Statevector


# VERIFY THAT THE TRAINED HQNN CHECKPOINT IS AVAILABLE

checkpoint_path = (
    "/content/best_hqnn_noisefree_finetuned.pth"
)

if not os.path.exists(
    checkpoint_path
):
    raise FileNotFoundError(
        "THE FINE-TUNED HQNN CHECKPOINT IS NOT AVAILABLE."
    )


# LOAD THE BEST TRAINED HQNN WEIGHTS

hqnn_model = hqnn_model.cpu()

hqnn_model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )
)

hqnn_model.eval()


# MOVE THE CLASSICAL FEATURE EXTRACTION COMPONENTS TO GPU IF AVAILABLE

comparison_device_classical = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

hqnn_model.features = (
    hqnn_model.features.to(
        comparison_device_classical
    )
)

hqnn_model.avgpool = (
    hqnn_model.avgpool.to(
        comparison_device_classical
    )
)

hqnn_model.feature_reduction = (
    hqnn_model.feature_reduction.to(
        comparison_device_classical
    )
)


# TAKE ONE REAL MRI IMAGE FROM THE TEST DATASET

sample_image, sample_label = (
    test_loader.dataset[0]
)

sample_image = (
    sample_image.unsqueeze(0).to(
        comparison_device_classical
    )
)


# GENERATE THE ACTUAL FOUR QUANTUM INPUT FEATURES

with torch.no_grad():

    sample_features = (
        hqnn_model.features(
            sample_image
        )
    )

    sample_features = (
        hqnn_model.avgpool(
            sample_features
        )
    )

    sample_features = torch.flatten(
        sample_features,
        1
    )

    sample_quantum_inputs = (
        hqnn_model.feature_reduction(
            sample_features
        )
    )

    sample_quantum_inputs = (
        torch.tanh(
            sample_quantum_inputs
        )
        * torch.pi
    )


sample_quantum_inputs = (
    sample_quantum_inputs
    .squeeze(0)
    .detach()
    .cpu()
)


# EXTRACT THE TRAINED 24 QUANTUM PARAMETERS

quantum_parameter_tensor = list(
    hqnn_model.quantum_layer.parameters()
)[0]

trained_quantum_weights = (
    quantum_parameter_tensor
    .detach()
    .cpu()
    .reshape(
        N_Q_LAYERS,
        N_QUBITS,
        3
    )
)


# CREATE AN INDEPENDENT PENNYLANE REFERENCE CIRCUIT

verification_device = qml.device(
    "default.qubit",
    wires=N_QUBITS
)


@qml.qnode(
    verification_device,
    interface="torch"
)
def verification_pennylane_circuit(
    inputs,
    weights
):

    # ENCODE THE FOUR REDUCED EFFICIENTNET FEATURES

    qml.AngleEmbedding(
        inputs,
        wires=range(
            N_QUBITS
        ),
        rotation="Y"
    )

    # APPLY THE TRAINED STRONGLY ENTANGLING LAYERS

    qml.StronglyEntanglingLayers(
        weights,
        wires=range(
            N_QUBITS
        )
    )

    # RETURN THE FOUR PAULI-Z EXPECTATION VALUES

    return [
        qml.expval(
            qml.PauliZ(
                qubit
            )
        )
        for qubit in range(
            N_QUBITS
        )
    ]


# CALCULATE EXACT PENNYLANE EXPECTATION VALUES

pennylane_output = (
    verification_pennylane_circuit(
        sample_quantum_inputs,
        trained_quantum_weights
    )
)

pennylane_expectations = np.array(
    [
        float(
            value.detach().cpu()
        )
        for value in pennylane_output
    ]
)


# BIND THE SAME INPUTS AND TRAINED WEIGHTS TO THE QISKIT CIRCUIT

parameter_bindings = {}

for index in range(
    N_QUBITS
):

    parameter_bindings[
        input_parameters[
            index
        ]
    ] = float(
        sample_quantum_inputs[
            index
        ]
    )


flattened_quantum_weights = (
    trained_quantum_weights
    .numpy()
    .reshape(-1)
)

for index in range(
    len(
        flattened_quantum_weights
    )
):

    parameter_bindings[
        weight_parameters[
            index
        ]
    ] = float(
        flattened_quantum_weights[
            index
        ]
    )


bound_qiskit_circuit = (
    reference_circuit.assign_parameters(
        parameter_bindings,
        inplace=False
    )
)


# CALCULATE EXACT QISKIT STATEVECTOR PROBABILITIES

qiskit_statevector = (
    Statevector.from_instruction(
        bound_qiskit_circuit
    )
)

qiskit_probabilities = (
    qiskit_statevector.probabilities_dict()
)


# CONVERT THE QISKIT STATEVECTOR TO PAULI-Z EXPECTATION VALUES

qiskit_expectations = np.zeros(
    N_QUBITS
)

for bitstring, probability in (
    qiskit_probabilities.items()
):

    bits = bitstring[::-1]

    for qubit in range(
        N_QUBITS
    ):

        if bits[
            qubit
        ] == "0":

            qiskit_expectations[
                qubit
            ] += probability

        else:

            qiskit_expectations[
                qubit
            ] -= probability


# COMPARE THE TWO IMPLEMENTATIONS

absolute_difference = np.abs(
    pennylane_expectations
    - qiskit_expectations
)

maximum_difference = np.max(
    absolute_difference
)


print(
    "TEST IMAGE TRUE CLASS:",
    sample_label
)

print(
    "\nQUANTUM INPUT VALUES:"
)

print(
    np.round(
        sample_quantum_inputs.numpy(),
        6
    )
)

print(
    "\nPENNYLANE EXPECTATION VALUES:"
)

print(
    np.round(
        pennylane_expectations,
        8
    )
)

print(
    "\nQISKIT EXPECTATION VALUES:"
)

print(
    np.round(
        qiskit_expectations,
        8
    )
)

print(
    "\nABSOLUTE DIFFERENCE:"
)

print(
    np.round(
        absolute_difference,
        10
    )
)

print(
    "\nMAXIMUM ABSOLUTE DIFFERENCE:",
    maximum_difference
)


# VERIFY CIRCUIT EQUIVALENCE

if maximum_difference < 1e-6:

    print(
        "\nPASS: PENNYLANE AND QISKIT CIRCUITS MATCH."
    )

else:

    print(
        "\nFAIL: THE TWO CIRCUITS DO NOT MATCH."
    )

TEST IMAGE TRUE CLASS: 1

QUANTUM INPUT VALUES:
[1.019473 0.883628 1.817462 0.672919]

PENNYLANE EXPECTATION VALUES:
[-0.68541293 -0.55061196  0.66086126  0.6074129 ]

QISKIT EXPECTATION VALUES:
[-0.68541284 -0.55061188  0.6608612   0.60741281]

ABSOLUTE DIFFERENCE:
[9.10e-08 8.09e-08 6.02e-08 9.74e-08]

MAXIMUM ABSOLUTE DIFFERENCE: 9.744504758657513e-08

PASS: PENNYLANE AND QISKIT CIRCUITS MATCH.


In [33]:
# UPLOAD THE TRAINED FINE-TUNED HQNN CHECKPOINT

from google.colab import files

uploaded = files.upload()

print(
    "Uploaded files:",
    list(uploaded.keys())
)

Saving BEST_HQNN.pth to BEST_HQNN.pth
Uploaded files: ['BEST_HQNN.pth']


In [34]:
# LOAD THE SAVED FINAL HQNN WEIGHTS

best_hqnn_state = torch.load(
    "/content/BEST_HQNN.pth",
    map_location="cpu",
    weights_only=True
)

hqnn_model.load_state_dict(
    best_hqnn_state,
    strict=True
)

print("BEST_HQNN.pth loaded successfully.")
print("Saved trained HQNN weights restored.")

BEST_HQNN.pth loaded successfully.
Saved trained HQNN weights restored.


In [35]:
# FREEZE THE TRAINED HQNN FOR CALIBRATION-NOISE EVALUATION

for param in hqnn_model.parameters():
    param.requires_grad = False

hqnn_model.eval()

print("HQNN locked for inference.")
print(
    "Trainable parameters:",
    sum(p.numel() for p in hqnn_model.parameters() if p.requires_grad)
)

HQNN locked for inference.
Trainable parameters: 0


In [36]:
# PREPARE THE LOCKED TRAINED HQNN FOR CALIBRATION-NOISE INFERENCE

# KEEP THE LOCKED MODEL ON CPU FOR CONSISTENT INFERENCE
hqnn_model = hqnn_model.cpu()
hqnn_model.eval()

# EXTRACT THE TRAINED 24 QUANTUM PARAMETERS
quantum_parameter_tensor = list(
    hqnn_model.quantum_layer.parameters()
)[0]

trained_quantum_weights = (
    quantum_parameter_tensor
    .detach()
    .cpu()
    .reshape(
        N_Q_LAYERS,
        N_QUBITS,
        3
    )
)

# FLATTEN THEM IN THE SAME ORDER USED BY THE QISKIT CIRCUIT
flattened_quantum_weights = (
    trained_quantum_weights
    .numpy()
    .reshape(-1)
)

print("Locked HQNN prepared for noisy inference.")
print("Quantum weight shape:", trained_quantum_weights.shape)
print("Number of quantum parameters:", len(flattened_quantum_weights))
print(
    "Trainable parameters remaining:",
    sum(
        p.numel()
        for p in hqnn_model.parameters()
        if p.requires_grad
    )
)

Locked HQNN prepared for noisy inference.
Quantum weight shape: torch.Size([2, 4, 3])
Number of quantum parameters: 24
Trainable parameters remaining: 0


In [37]:
# CREATE THE FIXED 40-IMAGE QUANTUM EVALUATION SUBSET
# 10 TEST IMAGES FROM EACH OF THE 4 CLASSES

import numpy as np
from torch.utils.data import Subset, DataLoader

SUBSET_PER_CLASS = 10
SUBSET_SEED = 42

rng = np.random.default_rng(SUBSET_SEED)

selected_indices = []

for class_name in classes:

    # FIND ALL TEST-SET INDICES FOR THIS CLASS
    class_indices = np.array([
        i
        for i, label in enumerate(test_labels)
        if label == class_name
    ])

    # RANDOMLY SELECT 10 WITHOUT REPLACEMENT
    chosen = rng.choice(
        class_indices,
        size=SUBSET_PER_CLASS,
        replace=False
    )

    selected_indices.extend(chosen.tolist())


# CREATE THE LOCKED 40-IMAGE SUBSET
quantum_test_subset = Subset(
    test_dataset,
    selected_indices
)

quantum_test_loader = DataLoader(
    quantum_test_subset,
    batch_size=1,
    shuffle=False
)


print("Fixed quantum evaluation subset created.")
print("Total images:", len(quantum_test_subset))

for class_name in classes:
    count = sum(
        test_labels[i] == class_name
        for i in selected_indices
    )
    print(class_name, ":", count)

Fixed quantum evaluation subset created.
Total images: 40
glioma : 10
meningioma : 10
notumor : 10
pituitary : 10


In [38]:
# EXTRACT THE 4 QUANTUM INPUTS FOR THE LOCKED 40-IMAGE SUBSET

import numpy as np
import torch

# USE GPU FOR THE CLASSICAL EFFICIENTNET PART IF AVAILABLE
feature_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

hqnn_model.features = hqnn_model.features.to(feature_device)
hqnn_model.avgpool = hqnn_model.avgpool.to(feature_device)
hqnn_model.feature_reduction = hqnn_model.feature_reduction.to(feature_device)

hqnn_model.eval()

quantum_inputs_40 = []
quantum_labels_40 = []

with torch.no_grad():

    for images, labels in quantum_test_loader:

        images = images.to(feature_device)

        # EFFICIENTNET-B0 FEATURE EXTRACTION
        features = hqnn_model.features(images)

        # GLOBAL AVERAGE POOLING
        features = hqnn_model.avgpool(features)

        # FLATTEN TO 1280 FEATURES
        features = torch.flatten(features, 1)

        # REDUCE 1280 -> 4
        quantum_inputs = hqnn_model.feature_reduction(features)

        # SAME ANGLE SCALING USED IN THE TRAINED HQNN
        quantum_inputs = torch.tanh(quantum_inputs) * torch.pi

        quantum_inputs_40.append(
            quantum_inputs.cpu().numpy()[0]
        )

        quantum_labels_40.append(
            labels.item()
        )


quantum_inputs_40 = np.array(quantum_inputs_40)
quantum_labels_40 = np.array(quantum_labels_40)

print("Quantum inputs extracted successfully.")
print("Quantum input shape:", quantum_inputs_40.shape)
print("Label shape:", quantum_labels_40.shape)

print("\nFirst MRI quantum input:")
print(quantum_inputs_40[0])

print("\nLabels:")
print(quantum_labels_40)

Quantum inputs extracted successfully.
Quantum input shape: (40, 4)
Label shape: (40,)

First MRI quantum input:
[0.09653189 2.920034   0.8836216  1.263368  ]

Labels:
[0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 3 3
 3 3 3]


In [39]:
# RUN THE LOCKED 40-IMAGE SUBSET THROUGH
# IDEAL FINITE-SHOT AND IBM CALIBRATION-NOISE SIMULATION

import numpy as np
import torch

from qiskit import ClassicalRegister

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ---------------------------------------------------------
# HELPER: CONVERT MEASUREMENT COUNTS TO 4 PAULI-Z EXPECTATIONS
# ---------------------------------------------------------

def counts_to_z_expectations(
    counts,
    number_of_qubits=4
):

    total_shots = sum(counts.values())

    expectations = np.zeros(
        number_of_qubits,
        dtype=np.float32
    )

    for bitstring, count in counts.items():

        clean_bitstring = bitstring.replace(
            " ",
            ""
        )

        # QISKIT BIT ORDER -> LOGICAL QUBIT ORDER
        bits = clean_bitstring[::-1]

        for qubit in range(
            number_of_qubits
        ):

            if bits[qubit] == "0":
                expectations[qubit] += count
            else:
                expectations[qubit] -= count

    expectations /= total_shots

    return expectations


# ---------------------------------------------------------
# PREPARE THE PARAMETERIZED TRANSPILED CIRCUIT
# ---------------------------------------------------------

base_transpiled_circuit = (
    selected_transpilation["circuit"]
)

# MAP PARAMETER NAMES IN THE TRANSPILED CIRCUIT
parameter_lookup = {
    parameter.name: parameter
    for parameter
    in base_transpiled_circuit.parameters
}


# ---------------------------------------------------------
# FUNCTION: EXECUTE ONE MRI QUANTUM INPUT
# ---------------------------------------------------------

def execute_quantum_input(
    quantum_input,
    simulator,
    seed
):

    parameter_bindings = {}

    # BIND THE 4 MRI-DERIVED QUANTUM INPUTS
    for index in range(N_QUBITS):

        parameter_bindings[
            parameter_lookup[f"x[{index}]"]
        ] = float(
            quantum_input[index]
        )

    # BIND THE 24 LOCKED TRAINED QUANTUM WEIGHTS
    for index in range(
        len(flattened_quantum_weights)
    ):

        parameter_bindings[
            parameter_lookup[f"w[{index}]"]
        ] = float(
            flattened_quantum_weights[index]
        )

    # INSERT THE ACTUAL TRAINED PARAMETERS
    circuit = (
        base_transpiled_circuit
        .assign_parameters(
            parameter_bindings,
            inplace=False
        )
        .copy()
    )

    # ADD MEASUREMENT REGISTER
    measurement_register = ClassicalRegister(
        N_QUBITS,
        "measurement"
    )

    circuit.add_register(
        measurement_register
    )

    # MEASURE LOGICAL QUBITS FROM THEIR FINAL PHYSICAL LOCATIONS
    for logical_qubit, physical_qubit in enumerate(
        LOGICAL_TO_PHYSICAL
    ):

        circuit.measure(
            physical_qubit,
            measurement_register[
                logical_qubit
            ]
        )

    # EXECUTE
    job = simulator.run(
        circuit,
        shots=EVALUATION_SHOTS,
        seed_simulator=seed
    )

    counts = job.result().get_counts()

    return counts_to_z_expectations(
        counts,
        N_QUBITS
    )


# ---------------------------------------------------------
# STORAGE
# ---------------------------------------------------------

ideal_predictions = []
ideal_probabilities = []

noisy_predictions = []
noisy_probabilities = []

ideal_expectations_all = []
noisy_expectations_all = []


# FINAL CLASSICAL 4 -> 4 CLASSIFIER
hqnn_model.classifier = (
    hqnn_model.classifier.cpu()
)

hqnn_model.classifier.eval()


# ---------------------------------------------------------
# RUN ALL 40 MRI IMAGES
# ---------------------------------------------------------

print(
    "Running 40-image ideal + calibration-noise experiment..."
)

for image_index in range(
    len(quantum_inputs_40)
):

    quantum_input = (
        quantum_inputs_40[
            image_index
        ]
    )

    # IDEAL FINITE-SHOT EXECUTION
    ideal_expectations = (
        execute_quantum_input(
            quantum_input,
            ideal_shot_simulator,
            seed=42
        )
    )

    # IBM CALIBRATION-NOISE EXECUTION
    noisy_expectations = (
        execute_quantum_input(
            quantum_input,
            noisy_ibm_simulator,
            seed=42
        )
    )

    ideal_expectations_all.append(
        ideal_expectations
    )

    noisy_expectations_all.append(
        noisy_expectations
    )

    # PASS QUANTUM OUTPUT THROUGH THE LOCKED FINAL CLASSIFIER
    ideal_tensor = torch.tensor(
        ideal_expectations,
        dtype=torch.float32
    ).unsqueeze(0)

    noisy_tensor = torch.tensor(
        noisy_expectations,
        dtype=torch.float32
    ).unsqueeze(0)

    with torch.no_grad():

        ideal_logits = (
            hqnn_model.classifier(
                ideal_tensor
            )
        )

        noisy_logits = (
            hqnn_model.classifier(
                noisy_tensor
            )
        )

        ideal_probs = torch.softmax(
            ideal_logits,
            dim=1
        )

        noisy_probs = torch.softmax(
            noisy_logits,
            dim=1
        )

    ideal_predictions.append(
        int(
            torch.argmax(
                ideal_logits,
                dim=1
            ).item()
        )
    )

    noisy_predictions.append(
        int(
            torch.argmax(
                noisy_logits,
                dim=1
            ).item()
        )
    )

    ideal_probabilities.append(
        ideal_probs.numpy()[0]
    )

    noisy_probabilities.append(
        noisy_probs.numpy()[0]
    )

    print(
        f"Completed image "
        f"{image_index + 1}/"
        f"{len(quantum_inputs_40)}"
    )


# ---------------------------------------------------------
# CONVERT TO NUMPY
# ---------------------------------------------------------

ideal_predictions = np.array(
    ideal_predictions
)

noisy_predictions = np.array(
    noisy_predictions
)

ideal_probabilities = np.array(
    ideal_probabilities
)

noisy_probabilities = np.array(
    noisy_probabilities
)

ideal_expectations_all = np.array(
    ideal_expectations_all
)

noisy_expectations_all = np.array(
    noisy_expectations_all
)


# ---------------------------------------------------------
# CALCULATE METRICS
# ---------------------------------------------------------

ideal_accuracy = accuracy_score(
    quantum_labels_40,
    ideal_predictions
)

ideal_precision = precision_score(
    quantum_labels_40,
    ideal_predictions,
    average="macro",
    zero_division=0
)

ideal_recall = recall_score(
    quantum_labels_40,
    ideal_predictions,
    average="macro",
    zero_division=0
)

ideal_f1 = f1_score(
    quantum_labels_40,
    ideal_predictions,
    average="macro",
    zero_division=0
)

ideal_auc = roc_auc_score(
    quantum_labels_40,
    ideal_probabilities,
    multi_class="ovr",
    average="macro"
)


noisy_accuracy = accuracy_score(
    quantum_labels_40,
    noisy_predictions
)

noisy_precision = precision_score(
    quantum_labels_40,
    noisy_predictions,
    average="macro",
    zero_division=0
)

noisy_recall = recall_score(
    quantum_labels_40,
    noisy_predictions,
    average="macro",
    zero_division=0
)

noisy_f1 = f1_score(
    quantum_labels_40,
    noisy_predictions,
    average="macro",
    zero_division=0
)

noisy_auc = roc_auc_score(
    quantum_labels_40,
    noisy_probabilities,
    multi_class="ovr",
    average="macro"
)


# ---------------------------------------------------------
# PRINT RESULTS
# ---------------------------------------------------------

print(
    "\nIDEAL FINITE-SHOT RESULTS"
)

print(
    f"Accuracy:        {ideal_accuracy:.4f}"
)

print(
    f"Macro Precision: {ideal_precision:.4f}"
)

print(
    f"Macro Recall:    {ideal_recall:.4f}"
)

print(
    f"Macro F1:        {ideal_f1:.4f}"
)

print(
    f"Macro ROC-AUC:   {ideal_auc:.4f}"
)


print(
    "\nIBM CALIBRATION-NOISE RESULTS"
)

print(
    f"Accuracy:        {noisy_accuracy:.4f}"
)

print(
    f"Macro Precision: {noisy_precision:.4f}"
)

print(
    f"Macro Recall:    {noisy_recall:.4f}"
)

print(
    f"Macro F1:        {noisy_f1:.4f}"
)

print(
    f"Macro ROC-AUC:   {noisy_auc:.4f}"
)

Running 40-image ideal + calibration-noise experiment...
Completed image 1/40
Completed image 2/40
Completed image 3/40
Completed image 4/40
Completed image 5/40
Completed image 6/40
Completed image 7/40
Completed image 8/40
Completed image 9/40
Completed image 10/40
Completed image 11/40
Completed image 12/40
Completed image 13/40
Completed image 14/40
Completed image 15/40
Completed image 16/40
Completed image 17/40
Completed image 18/40
Completed image 19/40
Completed image 20/40
Completed image 21/40
Completed image 22/40
Completed image 23/40
Completed image 24/40
Completed image 25/40
Completed image 26/40
Completed image 27/40
Completed image 28/40
Completed image 29/40
Completed image 30/40
Completed image 31/40
Completed image 32/40
Completed image 33/40
Completed image 34/40
Completed image 35/40
Completed image 36/40
Completed image 37/40
Completed image 38/40
Completed image 39/40
Completed image 40/40

IDEAL FINITE-SHOT RESULTS
Accuracy:        0.2500
Macro Precision: 0.06

In [40]:
# DIAGNOSE PENNYLANE VS QISKIT IDEAL EXECUTION
# DO NOT RUN THE 40-IMAGE EXPERIMENT AGAIN YET

print("Comparing trained PennyLane HQNN vs Qiskit ideal execution...\n")

for image_index in range(5):

    quantum_input = quantum_inputs_40[image_index]

    # --------------------------------------------------
    # ORIGINAL TRAINED PENNYLANE QUANTUM LAYER
    # --------------------------------------------------

    q_input_tensor = torch.tensor(
        quantum_input,
        dtype=torch.float32
    ).unsqueeze(0)

    with torch.no_grad():

        pennylane_expectations = (
            hqnn_model.quantum_layer(
                q_input_tensor
            )
        )

        pennylane_logits = (
            hqnn_model.classifier(
                pennylane_expectations
            )
        )

        pennylane_prediction = int(
            torch.argmax(
                pennylane_logits,
                dim=1
            ).item()
        )


    # --------------------------------------------------
    # CURRENT QISKIT IDEAL EXECUTION
    # --------------------------------------------------

    qiskit_expectations = execute_quantum_input(
        quantum_input,
        ideal_shot_simulator,
        seed=42
    )

    qiskit_tensor = torch.tensor(
        qiskit_expectations,
        dtype=torch.float32
    ).unsqueeze(0)

    with torch.no_grad():

        qiskit_logits = (
            hqnn_model.classifier(
                qiskit_tensor
            )
        )

        qiskit_prediction = int(
            torch.argmax(
                qiskit_logits,
                dim=1
            ).item()
        )


    # --------------------------------------------------
    # PRINT COMPARISON
    # --------------------------------------------------

    print("---------------------------------------")
    print("Image:", image_index + 1)
    print("True class:", quantum_labels_40[image_index])

    print(
        "PennyLane expectations:",
        np.round(
            pennylane_expectations.cpu().numpy()[0],
            4
        )
    )

    print(
        "Qiskit expectations:   ",
        np.round(
            qiskit_expectations,
            4
        )
    )

    print(
        "PennyLane prediction:",
        pennylane_prediction
    )

    print(
        "Qiskit prediction:   ",
        qiskit_prediction
    )

    print(
        "Maximum expectation difference:",
        np.max(
            np.abs(
                pennylane_expectations.cpu().numpy()[0]
                - qiskit_expectations
            )
        )
    )

Comparing trained PennyLane HQNN vs Qiskit ideal execution...

---------------------------------------
Image: 1
True class: 0
PennyLane expectations: [0.5119 0.121  0.1749 0.2449]
Qiskit expectations:    [1. 1. 1. 1.]
PennyLane prediction: 1
Qiskit prediction:    1
Maximum expectation difference: 0.878962
---------------------------------------
Image: 2
True class: 0
PennyLane expectations: [-0.8667  0.8027 -0.9936  0.8042]
Qiskit expectations:    [1. 1. 1. 1.]
PennyLane prediction: 0
Qiskit prediction:    1
Maximum expectation difference: 1.9936377
---------------------------------------
Image: 3
True class: 0
PennyLane expectations: [-0.9178  0.8153 -0.9371  0.8578]
Qiskit expectations:    [1. 1. 1. 1.]
PennyLane prediction: 0
Qiskit prediction:    1
Maximum expectation difference: 1.9370924
---------------------------------------
Image: 4
True class: 0
PennyLane expectations: [-0.4518  0.3571 -0.4187  0.7058]
Qiskit expectations:    [1. 1. 1. 1.]
PennyLane prediction: 0
Qiskit predi

In [41]:
# FIX THE QISKIT MEASUREMENT MAPPING
# MEASURE LOGICAL QUBITS BEFORE TRANSPILATION

from qiskit import QuantumCircuit, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager


# ---------------------------------------------------------
# CREATE A MEASURED VERSION OF THE ORIGINAL 4-QUBIT CIRCUIT
# ---------------------------------------------------------

measured_reference_circuit = reference_circuit.copy()

measurement_register = ClassicalRegister(
    N_QUBITS,
    "measurement"
)

measured_reference_circuit.add_register(
    measurement_register
)

# IMPORTANT:
# Measure LOGICAL qubits before transpilation.
# Qiskit will preserve the classical-bit relationship
# through layout and routing.
for logical_qubit in range(N_QUBITS):

    measured_reference_circuit.measure(
        logical_qubit,
        measurement_register[logical_qubit]
    )


# ---------------------------------------------------------
# TRANSPILE THE MEASURED CIRCUIT ON OUR SELECTED IBM REGION
# ---------------------------------------------------------

measurement_pass_manager = generate_preset_pass_manager(
    backend=ibm_backend,
    optimization_level=3,
    initial_layout=list(SELECTED_PHYSICAL_REGION),
    seed_transpiler=42
)

measured_transpiled_circuit = (
    measurement_pass_manager.run(
        measured_reference_circuit
    )
)


# ---------------------------------------------------------
# PARAMETER LOOKUP AFTER TRANSPILATION
# ---------------------------------------------------------

measured_parameter_lookup = {
    parameter.name: parameter
    for parameter
    in measured_transpiled_circuit.parameters
}


print("Corrected measured circuit created.")
print(
    "Circuit depth:",
    measured_transpiled_circuit.depth()
)
print(
    "Classical bits:",
    measured_transpiled_circuit.num_clbits
)
print(
    "Remaining parameters:",
    len(measured_transpiled_circuit.parameters)
)

Corrected measured circuit created.
Circuit depth: 62
Classical bits: 4
Remaining parameters: 28


In [43]:
# CORRECTED QISKIT EXECUTION FUNCTION

def execute_quantum_input_fixed(
    quantum_input,
    simulator,
    seed
):

    parameter_bindings = {}

    # BIND 4 MRI-DERIVED INPUT ANGLES
    for index in range(N_QUBITS):

        parameter_bindings[
            measured_parameter_lookup[
                f"x[{index}]"
            ]
        ] = float(
            quantum_input[index]
        )

    # BIND 24 TRAINED QUANTUM WEIGHTS
    for index in range(
        len(flattened_quantum_weights)
    ):

        parameter_bindings[
            measured_parameter_lookup[
                f"w[{index}]"
            ]
        ] = float(
            flattened_quantum_weights[index]
        )

    bound_circuit = (
        measured_transpiled_circuit
        .assign_parameters(
            parameter_bindings,
            inplace=False
        )
    )

    job = simulator.run(
        bound_circuit,
        shots=EVALUATION_SHOTS,
        seed_simulator=seed
    )

    counts = job.result().get_counts()

    return counts_to_z_expectations(
        counts,
        N_QUBITS
    )

In [44]:
# TEST THE FIX USING ONE MRI BEFORE RE-RUNNING ALL 40

quantum_input = quantum_inputs_40[0]

q_input_tensor = torch.tensor(
    quantum_input,
    dtype=torch.float32
).unsqueeze(0)


# ORIGINAL PENNYLANE HQNN
with torch.no_grad():

    pennylane_expectations = (
        hqnn_model.quantum_layer(
            q_input_tensor
        )
    )

pennylane_expectations_np = (
    pennylane_expectations
    .cpu()
    .numpy()[0]
)


# CORRECTED QISKIT IDEAL CIRCUIT
qiskit_fixed_expectations = (
    execute_quantum_input_fixed(
        quantum_input,
        ideal_shot_simulator,
        seed=42
    )
)


print("PennyLane expectations:")
print(
    np.round(
        pennylane_expectations_np,
        4
    )
)

print("\nCorrected Qiskit expectations:")
print(
    np.round(
        qiskit_fixed_expectations,
        4
    )
)

print("\nAbsolute difference:")
print(
    np.round(
        np.abs(
            pennylane_expectations_np
            - qiskit_fixed_expectations
        ),
        4
    )
)

print(
    "\nMaximum difference:",
    np.max(
        np.abs(
            pennylane_expectations_np
            - qiskit_fixed_expectations
        )
    )
)

PennyLane expectations:
[0.5119 0.121  0.1749 0.2449]

Corrected Qiskit expectations:
[0.5215 0.1504 0.209  0.2324]

Absolute difference:
[0.0096 0.0294 0.034  0.0125]

Maximum difference: 0.034046456


In [45]:
# RUN THE FIXED 40-IMAGE IDEAL VS IBM CALIBRATION-NOISE EXPERIMENT

ideal_predictions = []
ideal_probabilities = []

noisy_predictions = []
noisy_probabilities = []

ideal_expectations_all = []
noisy_expectations_all = []


hqnn_model.classifier = hqnn_model.classifier.cpu()
hqnn_model.classifier.eval()


print("Running corrected 40-image experiment...\n")


for image_index in range(len(quantum_inputs_40)):

    quantum_input = quantum_inputs_40[image_index]


    # IDEAL FINITE-SHOT SIMULATION
    ideal_expectations = execute_quantum_input_fixed(
        quantum_input,
        ideal_shot_simulator,
        seed=42
    )


    # IBM CALIBRATION-NOISE SIMULATION
    noisy_expectations = execute_quantum_input_fixed(
        quantum_input,
        noisy_ibm_simulator,
        seed=42
    )


    ideal_expectations_all.append(
        ideal_expectations
    )

    noisy_expectations_all.append(
        noisy_expectations
    )


    # CONVERT QUANTUM OUTPUTS TO TENSORS
    ideal_tensor = torch.tensor(
        ideal_expectations,
        dtype=torch.float32
    ).unsqueeze(0)

    noisy_tensor = torch.tensor(
        noisy_expectations,
        dtype=torch.float32
    ).unsqueeze(0)


    # FINAL LOCKED CLASSIFIER
    with torch.no_grad():

        ideal_logits = hqnn_model.classifier(
            ideal_tensor
        )

        noisy_logits = hqnn_model.classifier(
            noisy_tensor
        )

        ideal_probs = torch.softmax(
            ideal_logits,
            dim=1
        )

        noisy_probs = torch.softmax(
            noisy_logits,
            dim=1
        )


    ideal_predictions.append(
        int(
            torch.argmax(
                ideal_logits,
                dim=1
            ).item()
        )
    )

    noisy_predictions.append(
        int(
            torch.argmax(
                noisy_logits,
                dim=1
            ).item()
        )
    )

    ideal_probabilities.append(
        ideal_probs.cpu().numpy()[0]
    )

    noisy_probabilities.append(
        noisy_probs.cpu().numpy()[0]
    )


    print(
        f"Completed image {image_index + 1}/40"
    )


# CONVERT TO ARRAYS
ideal_predictions = np.array(
    ideal_predictions
)

noisy_predictions = np.array(
    noisy_predictions
)

ideal_probabilities = np.array(
    ideal_probabilities
)

noisy_probabilities = np.array(
    noisy_probabilities
)

ideal_expectations_all = np.array(
    ideal_expectations_all
)

noisy_expectations_all = np.array(
    noisy_expectations_all
)


# CALCULATE IDEAL METRICS
ideal_accuracy = accuracy_score(
    quantum_labels_40,
    ideal_predictions
)

ideal_precision = precision_score(
    quantum_labels_40,
    ideal_predictions,
    average="macro",
    zero_division=0
)

ideal_recall = recall_score(
    quantum_labels_40,
    ideal_predictions,
    average="macro",
    zero_division=0
)

ideal_f1 = f1_score(
    quantum_labels_40,
    ideal_predictions,
    average="macro",
    zero_division=0
)

ideal_auc = roc_auc_score(
    quantum_labels_40,
    ideal_probabilities,
    multi_class="ovr",
    average="macro"
)


# CALCULATE NOISY METRICS
noisy_accuracy = accuracy_score(
    quantum_labels_40,
    noisy_predictions
)

noisy_precision = precision_score(
    quantum_labels_40,
    noisy_predictions,
    average="macro",
    zero_division=0
)

noisy_recall = recall_score(
    quantum_labels_40,
    noisy_predictions,
    average="macro",
    zero_division=0
)

noisy_f1 = f1_score(
    quantum_labels_40,
    noisy_predictions,
    average="macro",
    zero_division=0
)

noisy_auc = roc_auc_score(
    quantum_labels_40,
    noisy_probabilities,
    multi_class="ovr",
    average="macro"
)


# PRINT RESULTS
print("\nIDEAL FINITE-SHOT RESULTS")

print(f"Accuracy:        {ideal_accuracy:.4f}")
print(f"Macro Precision: {ideal_precision:.4f}")
print(f"Macro Recall:    {ideal_recall:.4f}")
print(f"Macro F1:        {ideal_f1:.4f}")
print(f"Macro ROC-AUC:   {ideal_auc:.4f}")


print("\nIBM CALIBRATION-NOISE RESULTS")

print(f"Accuracy:        {noisy_accuracy:.4f}")
print(f"Macro Precision: {noisy_precision:.4f}")
print(f"Macro Recall:    {noisy_recall:.4f}")
print(f"Macro F1:        {noisy_f1:.4f}")
print(f"Macro ROC-AUC:   {noisy_auc:.4f}")


# ADDITIONAL DIRECT NOISE MEASUREMENT
mean_expectation_difference = np.mean(
    np.abs(
        ideal_expectations_all
        - noisy_expectations_all
    )
)

changed_predictions = np.sum(
    ideal_predictions
    != noisy_predictions
)

print(
    "\nMean absolute quantum-output difference:",
    round(
        float(mean_expectation_difference),
        4
    )
)

print(
    "Predictions changed by calibration noise:",
    f"{changed_predictions}/40"
)

Running corrected 40-image experiment...

Completed image 1/40
Completed image 2/40
Completed image 3/40
Completed image 4/40
Completed image 5/40
Completed image 6/40
Completed image 7/40
Completed image 8/40
Completed image 9/40
Completed image 10/40
Completed image 11/40
Completed image 12/40
Completed image 13/40
Completed image 14/40
Completed image 15/40
Completed image 16/40
Completed image 17/40
Completed image 18/40
Completed image 19/40
Completed image 20/40
Completed image 21/40
Completed image 22/40
Completed image 23/40
Completed image 24/40
Completed image 25/40
Completed image 26/40
Completed image 27/40
Completed image 28/40
Completed image 29/40
Completed image 30/40
Completed image 31/40
Completed image 32/40
Completed image 33/40
Completed image 34/40
Completed image 35/40
Completed image 36/40
Completed image 37/40
Completed image 38/40
Completed image 39/40
Completed image 40/40

IDEAL FINITE-SHOT RESULTS
Accuracy:        0.8500
Macro Precision: 0.8640
Macro Recall

In [46]:
# ============================================================
# FINAL CONTROLLED CALIBRATION-NOISE EXPERIMENT
# SAME 40 MRI IMAGES
# SAME LOCKED BEST_HQNN WEIGHTS
# SAME 1024 SHOTS
# THREE REPEATS: SEEDS 42, 43, 44
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ------------------------------------------------------------
# KEEP THE TRAINED QUANTUM LAYER AND CLASSIFIER LOCKED ON CPU
# ------------------------------------------------------------

hqnn_model.quantum_layer = hqnn_model.quantum_layer.cpu()
hqnn_model.classifier = hqnn_model.classifier.cpu()

hqnn_model.quantum_layer.eval()
hqnn_model.classifier.eval()

for param in hqnn_model.parameters():
    param.requires_grad = False


# ------------------------------------------------------------
# HELPER FUNCTION FOR METRICS
# ------------------------------------------------------------

def calculate_metrics(
    true_labels,
    predictions,
    probabilities
):

    return {
        "accuracy": accuracy_score(
            true_labels,
            predictions
        ),

        "precision": precision_score(
            true_labels,
            predictions,
            average="macro",
            zero_division=0
        ),

        "recall": recall_score(
            true_labels,
            predictions,
            average="macro",
            zero_division=0
        ),

        "f1": f1_score(
            true_labels,
            predictions,
            average="macro",
            zero_division=0
        ),

        "auc": roc_auc_score(
            true_labels,
            probabilities,
            multi_class="ovr",
            average="macro"
        )
    }


# ============================================================
# PART 1 — EXACT NOISE-FREE PENNYLANE CONTROL
# ============================================================

print("Running exact noise-free PennyLane control...\n")

exact_predictions = []
exact_probabilities = []
exact_expectations_all = []


for image_index in range(
    len(quantum_inputs_40)
):

    quantum_input_tensor = torch.tensor(
        quantum_inputs_40[image_index],
        dtype=torch.float32
    ).unsqueeze(0)


    with torch.no_grad():

        # EXACT ANALYTICAL QUANTUM OUTPUT
        exact_expectations = (
            hqnn_model.quantum_layer(
                quantum_input_tensor
            )
        )

        # FINAL LOCKED CLASSIFIER
        exact_logits = (
            hqnn_model.classifier(
                exact_expectations
            )
        )

        exact_probs = torch.softmax(
            exact_logits,
            dim=1
        )


    exact_expectations_all.append(
        exact_expectations
        .cpu()
        .numpy()[0]
    )

    exact_predictions.append(
        int(
            torch.argmax(
                exact_logits,
                dim=1
            ).item()
        )
    )

    exact_probabilities.append(
        exact_probs
        .cpu()
        .numpy()[0]
    )


exact_expectations_all = np.array(
    exact_expectations_all
)

exact_predictions = np.array(
    exact_predictions
)

exact_probabilities = np.array(
    exact_probabilities
)


exact_metrics = calculate_metrics(
    quantum_labels_40,
    exact_predictions,
    exact_probabilities
)


print("EXACT NOISE-FREE PENNYLANE RESULTS")

print(
    f"Accuracy:        "
    f"{exact_metrics['accuracy']:.4f}"
)

print(
    f"Macro Precision: "
    f"{exact_metrics['precision']:.4f}"
)

print(
    f"Macro Recall:    "
    f"{exact_metrics['recall']:.4f}"
)

print(
    f"Macro F1:        "
    f"{exact_metrics['f1']:.4f}"
)

print(
    f"Macro ROC-AUC:   "
    f"{exact_metrics['auc']:.4f}"
)


# ============================================================
# PART 2 — THREE IDEAL + CALIBRATION-NOISE REPEATS
# ============================================================

REPEAT_SEEDS = [
    42,
    43,
    44
]

ideal_repeat_results = []
noisy_repeat_results = []


for repeat_number, seed in enumerate(
    REPEAT_SEEDS,
    start=1
):

    print(
        "\n================================================"
    )

    print(
        f"RUN {repeat_number}/3 — "
        f"SEED {seed}"
    )

    print(
        "================================================"
    )


    ideal_predictions = []
    ideal_probabilities = []
    ideal_expectations_all = []

    noisy_predictions = []
    noisy_probabilities = []
    noisy_expectations_all = []


    # --------------------------------------------------------
    # RUN ALL 40 MRI IMAGES
    # --------------------------------------------------------

    for image_index in range(
        len(quantum_inputs_40)
    ):

        quantum_input = (
            quantum_inputs_40[
                image_index
            ]
        )


        # ----------------------------------------------------
        # IDEAL 1024-SHOT SIMULATION
        # ----------------------------------------------------

        ideal_expectations = (
            execute_quantum_input_fixed(
                quantum_input,
                ideal_shot_simulator,
                seed=seed
            )
        )


        # ----------------------------------------------------
        # IBM CALIBRATION-NOISE 1024-SHOT SIMULATION
        # ----------------------------------------------------

        noisy_expectations = (
            execute_quantum_input_fixed(
                quantum_input,
                noisy_ibm_simulator,
                seed=seed
            )
        )


        ideal_expectations_all.append(
            ideal_expectations
        )

        noisy_expectations_all.append(
            noisy_expectations
        )


        # ----------------------------------------------------
        # PASS QUANTUM OUTPUT THROUGH SAME LOCKED CLASSIFIER
        # ----------------------------------------------------

        ideal_tensor = torch.tensor(
            ideal_expectations,
            dtype=torch.float32
        ).unsqueeze(0)

        noisy_tensor = torch.tensor(
            noisy_expectations,
            dtype=torch.float32
        ).unsqueeze(0)


        with torch.no_grad():

            ideal_logits = (
                hqnn_model.classifier(
                    ideal_tensor
                )
            )

            noisy_logits = (
                hqnn_model.classifier(
                    noisy_tensor
                )
            )


            ideal_probs = torch.softmax(
                ideal_logits,
                dim=1
            )

            noisy_probs = torch.softmax(
                noisy_logits,
                dim=1
            )


        ideal_predictions.append(
            int(
                torch.argmax(
                    ideal_logits,
                    dim=1
                ).item()
            )
        )

        noisy_predictions.append(
            int(
                torch.argmax(
                    noisy_logits,
                    dim=1
                ).item()
            )
        )


        ideal_probabilities.append(
            ideal_probs
            .cpu()
            .numpy()[0]
        )

        noisy_probabilities.append(
            noisy_probs
            .cpu()
            .numpy()[0]
        )


    # --------------------------------------------------------
    # CONVERT TO NUMPY
    # --------------------------------------------------------

    ideal_predictions = np.array(
        ideal_predictions
    )

    noisy_predictions = np.array(
        noisy_predictions
    )

    ideal_probabilities = np.array(
        ideal_probabilities
    )

    noisy_probabilities = np.array(
        noisy_probabilities
    )

    ideal_expectations_all = np.array(
        ideal_expectations_all
    )

    noisy_expectations_all = np.array(
        noisy_expectations_all
    )


    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    ideal_metrics = calculate_metrics(
        quantum_labels_40,
        ideal_predictions,
        ideal_probabilities
    )

    noisy_metrics = calculate_metrics(
        quantum_labels_40,
        noisy_predictions,
        noisy_probabilities
    )


    # --------------------------------------------------------
    # DIRECT QUANTUM NOISE EFFECT
    # --------------------------------------------------------

    mean_quantum_difference = np.mean(
        np.abs(
            ideal_expectations_all
            - noisy_expectations_all
        )
    )


    changed_predictions = int(
        np.sum(
            ideal_predictions
            != noisy_predictions
        )
    )


    # --------------------------------------------------------
    # SAVE THIS REPEAT
    # --------------------------------------------------------

    ideal_repeat_results.append(
        {
            **ideal_metrics,

            "quantum_difference":
                np.mean(
                    np.abs(
                        exact_expectations_all
                        - ideal_expectations_all
                    )
                )
        }
    )


    noisy_repeat_results.append(
        {
            **noisy_metrics,

            "quantum_difference":
                mean_quantum_difference,

            "changed_predictions":
                changed_predictions
        }
    )


    # --------------------------------------------------------
    # PRINT THIS REPEAT
    # --------------------------------------------------------

    print(
        "\nIDEAL FINITE-SHOT"
    )

    print(
        f"Accuracy: "
        f"{ideal_metrics['accuracy']:.4f}"
    )

    print(
        f"Macro F1: "
        f"{ideal_metrics['f1']:.4f}"
    )

    print(
        f"ROC-AUC: "
        f"{ideal_metrics['auc']:.4f}"
    )


    print(
        "\nIBM CALIBRATION-NOISE"
    )

    print(
        f"Accuracy: "
        f"{noisy_metrics['accuracy']:.4f}"
    )

    print(
        f"Macro F1: "
        f"{noisy_metrics['f1']:.4f}"
    )

    print(
        f"ROC-AUC: "
        f"{noisy_metrics['auc']:.4f}"
    )


    print(
        "\nMean |Ideal - Noisy| quantum difference:",
        round(
            float(
                mean_quantum_difference
            ),
            4
        )
    )

    print(
        "Predictions changed:",
        f"{changed_predictions}/40"
    )


# ============================================================
# PART 3 — FINAL MEAN ± STANDARD DEVIATION
# ============================================================

ideal_accuracies = np.array([
    result["accuracy"]
    for result in ideal_repeat_results
])

noisy_accuracies = np.array([
    result["accuracy"]
    for result in noisy_repeat_results
])


ideal_f1_scores = np.array([
    result["f1"]
    for result in ideal_repeat_results
])

noisy_f1_scores = np.array([
    result["f1"]
    for result in noisy_repeat_results
])


ideal_auc_scores = np.array([
    result["auc"]
    for result in ideal_repeat_results
])

noisy_auc_scores = np.array([
    result["auc"]
    for result in noisy_repeat_results
])


noise_differences = np.array([
    result["quantum_difference"]
    for result in noisy_repeat_results
])


prediction_changes = np.array([
    result["changed_predictions"]
    for result in noisy_repeat_results
])


print(
    "\n\n================================================"
)

print(
    "FINAL THREE-RUN SUMMARY"
)

print(
    "================================================"
)


print(
    "\nEXACT NOISE-FREE PENNYLANE"
)

print(
    f"Accuracy: "
    f"{exact_metrics['accuracy']:.4f}"
)

print(
    f"Macro F1: "
    f"{exact_metrics['f1']:.4f}"
)

print(
    f"ROC-AUC: "
    f"{exact_metrics['auc']:.4f}"
)


print(
    "\nIDEAL 1024-SHOT SIMULATION"
)

print(
    f"Accuracy: "
    f"{ideal_accuracies.mean():.4f} "
    f"± {ideal_accuracies.std(ddof=1):.4f}"
)

print(
    f"Macro F1: "
    f"{ideal_f1_scores.mean():.4f} "
    f"± {ideal_f1_scores.std(ddof=1):.4f}"
)

print(
    f"ROC-AUC: "
    f"{ideal_auc_scores.mean():.4f} "
    f"± {ideal_auc_scores.std(ddof=1):.4f}"
)


print(
    "\nIBM CALIBRATION-NOISE 1024-SHOT SIMULATION"
)

print(
    f"Accuracy: "
    f"{noisy_accuracies.mean():.4f} "
    f"± {noisy_accuracies.std(ddof=1):.4f}"
)

print(
    f"Macro F1: "
    f"{noisy_f1_scores.mean():.4f} "
    f"± {noisy_f1_scores.std(ddof=1):.4f}"
)

print(
    f"ROC-AUC: "
    f"{noisy_auc_scores.mean():.4f} "
    f"± {noisy_auc_scores.std(ddof=1):.4f}"
)


print(
    "\nNOISE EFFECT"
)

print(
    "Mean absolute ideal-vs-noisy "
    "quantum-output difference:"
)

print(
    f"{noise_differences.mean():.4f} "
    f"± {noise_differences.std(ddof=1):.4f}"
)

print(
    "Mean number of changed predictions:"
)

print(
    f"{prediction_changes.mean():.2f}/40"
)

Running exact noise-free PennyLane control...

EXACT NOISE-FREE PENNYLANE RESULTS
Accuracy:        0.8500
Macro Precision: 0.8640
Macro Recall:    0.8500
Macro F1:        0.8456
Macro ROC-AUC:   0.9658

RUN 1/3 — SEED 42

IDEAL FINITE-SHOT
Accuracy: 0.8500
Macro F1: 0.8456
ROC-AUC: 0.9650

IBM CALIBRATION-NOISE
Accuracy: 0.8750
Macro F1: 0.8713
ROC-AUC: 0.9658

Mean |Ideal - Noisy| quantum difference: 0.039
Predictions changed: 1/40

RUN 2/3 — SEED 43

IDEAL FINITE-SHOT
Accuracy: 0.8250
Macro F1: 0.8166
ROC-AUC: 0.9658

IBM CALIBRATION-NOISE
Accuracy: 0.8500
Macro F1: 0.8456
ROC-AUC: 0.9658

Mean |Ideal - Noisy| quantum difference: 0.039
Predictions changed: 1/40

RUN 3/3 — SEED 44

IDEAL FINITE-SHOT
Accuracy: 0.8750
Macro F1: 0.8713
ROC-AUC: 0.9658

IBM CALIBRATION-NOISE
Accuracy: 0.8750
Macro F1: 0.8713
ROC-AUC: 0.9675

Mean |Ideal - Noisy| quantum difference: 0.0365
Predictions changed: 0/40


FINAL THREE-RUN SUMMARY

EXACT NOISE-FREE PENNYLANE
Accuracy: 0.8500
Macro F1: 0.8456
ROC-

In [47]:
# ============================================================
# EXTRACT QUANTUM INPUTS FOR THE FULL 1,080-IMAGE TEST SET
# ============================================================

import numpy as np
import torch

feature_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

hqnn_model.features = hqnn_model.features.to(feature_device)
hqnn_model.avgpool = hqnn_model.avgpool.to(feature_device)
hqnn_model.feature_reduction = hqnn_model.feature_reduction.to(feature_device)

hqnn_model.eval()

full_quantum_inputs = []
full_test_labels = []

print("Extracting quantum inputs from all test MRIs...")

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(feature_device)

        # EFFICIENTNET-B0
        features = hqnn_model.features(images)

        # GLOBAL AVERAGE POOL
        features = hqnn_model.avgpool(features)

        # 1280 FEATURES
        features = torch.flatten(
            features,
            1
        )

        # 1280 -> 4
        quantum_inputs = (
            hqnn_model.feature_reduction(
                features
            )
        )

        # SAME QUANTUM ANGLE SCALING AS TRAINED HQNN
        quantum_inputs = (
            torch.tanh(
                quantum_inputs
            )
            * torch.pi
        )

        full_quantum_inputs.append(
            quantum_inputs
            .cpu()
            .numpy()
        )

        full_test_labels.append(
            labels
            .cpu()
            .numpy()
        )


full_quantum_inputs = np.concatenate(
    full_quantum_inputs,
    axis=0
)

full_test_labels = np.concatenate(
    full_test_labels,
    axis=0
)


print(
    "Quantum input shape:",
    full_quantum_inputs.shape
)

print(
    "Label shape:",
    full_test_labels.shape
)

print(
    "Class counts:",
    np.bincount(
        full_test_labels
    )
)

Extracting quantum inputs from all test MRIs...
Quantum input shape: (1080, 4)
Label shape: (1080,)
Class counts: [270 270 270 270]


In [48]:
# ============================================================
# FULL 1,080-IMAGE IDEAL VS IBM CALIBRATION-NOISE EXPERIMENT
#
# SAME:
# - BEST_HQNN weights
# - 1,080 test images
# - quantum circuit
# - final classifier
# - 1024 shots
#
# ONLY DIFFERENCE:
# ideal simulator vs IBM calibration-derived noisy simulator
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


# ------------------------------------------------------------
# LOCK FINAL CLASSIFIER
# ------------------------------------------------------------

hqnn_model.classifier = (
    hqnn_model.classifier.cpu()
)

hqnn_model.classifier.eval()

for param in hqnn_model.parameters():
    param.requires_grad = False


# ------------------------------------------------------------
# BUILD ALL 1,080 PARAMETER-BOUND QUANTUM CIRCUITS
# ------------------------------------------------------------

print(
    "Building 1,080 trained HQNN quantum circuits..."
)

full_bound_circuits = []


for image_index in range(
    len(full_quantum_inputs)
):

    quantum_input = (
        full_quantum_inputs[
            image_index
        ]
    )

    parameter_bindings = {}


    # 4 MRI-DERIVED QUANTUM INPUTS
    for index in range(
        N_QUBITS
    ):

        parameter_bindings[
            measured_parameter_lookup[
                f"x[{index}]"
            ]
        ] = float(
            quantum_input[index]
        )


    # 24 LOCKED TRAINED QUANTUM WEIGHTS
    for index in range(
        len(
            flattened_quantum_weights
        )
    ):

        parameter_bindings[
            measured_parameter_lookup[
                f"w[{index}]"
            ]
        ] = float(
            flattened_quantum_weights[
                index
            ]
        )


    bound_circuit = (
        measured_transpiled_circuit
        .assign_parameters(
            parameter_bindings,
            inplace=False
        )
    )


    full_bound_circuits.append(
        bound_circuit
    )


print(
    "Circuits prepared:",
    len(full_bound_circuits)
)


# ------------------------------------------------------------
# IDEAL FINITE-SHOT SIMULATION
# ------------------------------------------------------------

print(
    "\nRunning IDEAL 1024-shot simulation..."
)

ideal_job_full = (
    ideal_shot_simulator.run(
        full_bound_circuits,
        shots=EVALUATION_SHOTS,
        seed_simulator=4200
    )
)

ideal_result_full = (
    ideal_job_full.result()
)

print(
    "Ideal simulation complete."
)


# ------------------------------------------------------------
# IBM CALIBRATION-NOISE SIMULATION
# ------------------------------------------------------------

print(
    "\nRunning IBM calibration-noise simulation..."
)

noisy_job_full = (
    noisy_ibm_simulator.run(
        full_bound_circuits,
        shots=EVALUATION_SHOTS,
        seed_simulator=4200
    )
)

noisy_result_full = (
    noisy_job_full.result()
)

print(
    "Calibration-noise simulation complete."
)


# ------------------------------------------------------------
# CONVERT ALL COUNTS TO QUANTUM EXPECTATION VALUES
# ------------------------------------------------------------

ideal_expectations_full = []
noisy_expectations_full = []


for image_index in range(
    len(full_bound_circuits)
):

    ideal_counts = (
        ideal_result_full.get_counts(
            image_index
        )
    )

    noisy_counts = (
        noisy_result_full.get_counts(
            image_index
        )
    )


    ideal_expectations = (
        counts_to_z_expectations(
            ideal_counts,
            N_QUBITS
        )
    )

    noisy_expectations = (
        counts_to_z_expectations(
            noisy_counts,
            N_QUBITS
        )
    )


    ideal_expectations_full.append(
        ideal_expectations
    )

    noisy_expectations_full.append(
        noisy_expectations
    )


ideal_expectations_full = np.array(
    ideal_expectations_full
)

noisy_expectations_full = np.array(
    noisy_expectations_full
)


print(
    "\nIdeal expectation shape:",
    ideal_expectations_full.shape
)

print(
    "Noisy expectation shape:",
    noisy_expectations_full.shape
)


# ------------------------------------------------------------
# PASS BOTH THROUGH THE SAME LOCKED 4 -> 4 CLASSIFIER
# ------------------------------------------------------------

ideal_tensor_full = torch.tensor(
    ideal_expectations_full,
    dtype=torch.float32
)

noisy_tensor_full = torch.tensor(
    noisy_expectations_full,
    dtype=torch.float32
)


with torch.no_grad():

    ideal_logits_full = (
        hqnn_model.classifier(
            ideal_tensor_full
        )
    )

    noisy_logits_full = (
        hqnn_model.classifier(
            noisy_tensor_full
        )
    )


    ideal_probabilities_full = (
        torch.softmax(
            ideal_logits_full,
            dim=1
        )
        .cpu()
        .numpy()
    )

    noisy_probabilities_full = (
        torch.softmax(
            noisy_logits_full,
            dim=1
        )
        .cpu()
        .numpy()
    )


ideal_predictions_full = (
    torch.argmax(
        ideal_logits_full,
        dim=1
    )
    .cpu()
    .numpy()
)

noisy_predictions_full = (
    torch.argmax(
        noisy_logits_full,
        dim=1
    )
    .cpu()
    .numpy()
)


# ------------------------------------------------------------
# METRIC FUNCTION
# ------------------------------------------------------------

def calculate_full_metrics(
    labels,
    predictions,
    probabilities
):

    return {

        "accuracy":
            accuracy_score(
                labels,
                predictions
            ),

        "precision":
            precision_score(
                labels,
                predictions,
                average="macro",
                zero_division=0
            ),

        "recall":
            recall_score(
                labels,
                predictions,
                average="macro",
                zero_division=0
            ),

        "f1":
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0
            ),

        "auc":
            roc_auc_score(
                labels,
                probabilities,
                multi_class="ovr",
                average="macro"
            )
    }


ideal_full_metrics = (
    calculate_full_metrics(
        full_test_labels,
        ideal_predictions_full,
        ideal_probabilities_full
    )
)

noisy_full_metrics = (
    calculate_full_metrics(
        full_test_labels,
        noisy_predictions_full,
        noisy_probabilities_full
    )
)


# ------------------------------------------------------------
# DIRECT NOISE EFFECT
# ------------------------------------------------------------

mean_noise_shift_full = np.mean(
    np.abs(
        ideal_expectations_full
        - noisy_expectations_full
    )
)

changed_predictions_full = np.sum(
    ideal_predictions_full
    != noisy_predictions_full
)


# DID NOISE HELP OR HURT EACH CHANGED SAMPLE?
ideal_correct = (
    ideal_predictions_full
    == full_test_labels
)

noisy_correct = (
    noisy_predictions_full
    == full_test_labels
)


noise_fixed_errors = np.sum(
    (~ideal_correct)
    &
    noisy_correct
)

noise_created_errors = np.sum(
    ideal_correct
    &
    (~noisy_correct)
)


# ------------------------------------------------------------
# PRINT FINAL RESULTS
# ------------------------------------------------------------

print(
    "\n========================================"
)

print(
    "FULL 1,080-IMAGE RESULTS"
)

print(
    "========================================"
)


print(
    "\nIDEAL 1024-SHOT SIMULATION"
)

print(
    f"Accuracy:        "
    f"{ideal_full_metrics['accuracy']:.4f}"
)

print(
    f"Macro Precision: "
    f"{ideal_full_metrics['precision']:.4f}"
)

print(
    f"Macro Recall:    "
    f"{ideal_full_metrics['recall']:.4f}"
)

print(
    f"Macro F1:        "
    f"{ideal_full_metrics['f1']:.4f}"
)

print(
    f"Macro ROC-AUC:   "
    f"{ideal_full_metrics['auc']:.4f}"
)


print(
    "\nIBM CALIBRATION-NOISE "
    "1024-SHOT SIMULATION"
)

print(
    f"Accuracy:        "
    f"{noisy_full_metrics['accuracy']:.4f}"
)

print(
    f"Macro Precision: "
    f"{noisy_full_metrics['precision']:.4f}"
)

print(
    f"Macro Recall:    "
    f"{noisy_full_metrics['recall']:.4f}"
)

print(
    f"Macro F1:        "
    f"{noisy_full_metrics['f1']:.4f}"
)

print(
    f"Macro ROC-AUC:   "
    f"{noisy_full_metrics['auc']:.4f}"
)


print(
    "\nNOISE EFFECT"
)

print(
    "Mean absolute quantum-output shift:",
    round(
        float(
            mean_noise_shift_full
        ),
        4
    )
)

print(
    "Predictions changed:",
    f"{changed_predictions_full}/1080"
)

print(
    "Errors FIXED by noise:",
    int(
        noise_fixed_errors
    )
)

print(
    "Errors CREATED by noise:",
    int(
        noise_created_errors
    )
)


print(
    "\nIDEAL CONFUSION MATRIX:"
)

print(
    confusion_matrix(
        full_test_labels,
        ideal_predictions_full
    )
)


print(
    "\nCALIBRATION-NOISE CONFUSION MATRIX:"
)

print(
    confusion_matrix(
        full_test_labels,
        noisy_predictions_full
    )
)

Building 1,080 trained HQNN quantum circuits...
Circuits prepared: 1080

Running IDEAL 1024-shot simulation...
Ideal simulation complete.

Running IBM calibration-noise simulation...
Calibration-noise simulation complete.

Ideal expectation shape: (1080, 4)
Noisy expectation shape: (1080, 4)

FULL 1,080-IMAGE RESULTS

IDEAL 1024-SHOT SIMULATION
Accuracy:        0.8917
Macro Precision: 0.8949
Macro Recall:    0.8917
Macro F1:        0.8902
Macro ROC-AUC:   0.9816

IBM CALIBRATION-NOISE 1024-SHOT SIMULATION
Accuracy:        0.8889
Macro Precision: 0.8924
Macro Recall:    0.8889
Macro F1:        0.8875
Macro ROC-AUC:   0.9812

NOISE EFFECT
Mean absolute quantum-output shift: 0.0383
Predictions changed: 5/1080
Errors FIXED by noise: 0
Errors CREATED by noise: 3

IDEAL CONFUSION MATRIX:
[[211  34  15  10]
 [  5 228  16  21]
 [  3   2 265   0]
 [  1  10   0 259]]

CALIBRATION-NOISE CONFUSION MATRIX:
[[210  37  14   9]
 [  5 227  16  22]
 [  3   3 264   0]
 [  1  10   0 259]]


In [49]:
# ============================================================
# EXACT NOISE-FREE PENNYLANE CONTROL — FULL 1,080 TEST IMAGES
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


hqnn_model.quantum_layer = hqnn_model.quantum_layer.cpu()
hqnn_model.classifier = hqnn_model.classifier.cpu()

hqnn_model.quantum_layer.eval()
hqnn_model.classifier.eval()


exact_expectations_full = []
exact_predictions_full = []
exact_probabilities_full = []


print("Running exact PennyLane control on 1,080 images...")


with torch.no_grad():

    for image_index in range(
        len(full_quantum_inputs)
    ):

        quantum_input = torch.tensor(
            full_quantum_inputs[image_index],
            dtype=torch.float32
        ).unsqueeze(0)


        # EXACT NOISE-FREE QUANTUM CIRCUIT
        exact_expectations = (
            hqnn_model.quantum_layer(
                quantum_input
            )
        )


        # SAME LOCKED FINAL CLASSIFIER
        exact_logits = (
            hqnn_model.classifier(
                exact_expectations
            )
        )


        exact_probs = torch.softmax(
            exact_logits,
            dim=1
        )


        exact_expectations_full.append(
            exact_expectations
            .cpu()
            .numpy()[0]
        )


        exact_predictions_full.append(
            int(
                torch.argmax(
                    exact_logits,
                    dim=1
                ).item()
            )
        )


        exact_probabilities_full.append(
            exact_probs
            .cpu()
            .numpy()[0]
        )


exact_expectations_full = np.array(
    exact_expectations_full
)

exact_predictions_full = np.array(
    exact_predictions_full
)

exact_probabilities_full = np.array(
    exact_probabilities_full
)


# ============================================================
# METRICS
# ============================================================

exact_accuracy_full = accuracy_score(
    full_test_labels,
    exact_predictions_full
)

exact_precision_full = precision_score(
    full_test_labels,
    exact_predictions_full,
    average="macro",
    zero_division=0
)

exact_recall_full = recall_score(
    full_test_labels,
    exact_predictions_full,
    average="macro",
    zero_division=0
)

exact_f1_full = f1_score(
    full_test_labels,
    exact_predictions_full,
    average="macro",
    zero_division=0
)

exact_auc_full = roc_auc_score(
    full_test_labels,
    exact_probabilities_full,
    multi_class="ovr",
    average="macro"
)


print(
    "\n========================================"
)

print(
    "EXACT NOISE-FREE PENNYLANE — 1,080 IMAGES"
)

print(
    "========================================"
)

print(
    f"Accuracy:        {exact_accuracy_full:.4f}"
)

print(
    f"Macro Precision: {exact_precision_full:.4f}"
)

print(
    f"Macro Recall:    {exact_recall_full:.4f}"
)

print(
    f"Macro F1:        {exact_f1_full:.4f}"
)

print(
    f"Macro ROC-AUC:   {exact_auc_full:.4f}"
)


# ============================================================
# DECOMPOSE THE PERFORMANCE LOSS
# ============================================================

print(
    "\n========================================"
)

print(
    "PERFORMANCE DECOMPOSITION"
)

print(
    "========================================"
)


print(
    "\nExact noise-free accuracy:",
    f"{exact_accuracy_full:.4f}"
)

print(
    "Ideal 1024-shot accuracy:",
    f"{ideal_full_metrics['accuracy']:.4f}"
)

print(
    "Calibration-noise accuracy:",
    f"{noisy_full_metrics['accuracy']:.4f}"
)


print(
    "\nLoss from finite-shot sampling:",
    f"{exact_accuracy_full - ideal_full_metrics['accuracy']:.4f}"
)

print(
    "Additional loss from calibration noise:",
    f"{ideal_full_metrics['accuracy'] - noisy_full_metrics['accuracy']:.4f}"
)

print(
    "Total loss from exact to noisy:",
    f"{exact_accuracy_full - noisy_full_metrics['accuracy']:.4f}"
)

Running exact PennyLane control on 1,080 images...

EXACT NOISE-FREE PENNYLANE — 1,080 IMAGES
Accuracy:        0.8889
Macro Precision: 0.8921
Macro Recall:    0.8889
Macro F1:        0.8873
Macro ROC-AUC:   0.9818

PERFORMANCE DECOMPOSITION

Exact noise-free accuracy: 0.8889
Ideal 1024-shot accuracy: 0.8917
Calibration-noise accuracy: 0.8889

Loss from finite-shot sampling: -0.0028
Additional loss from calibration noise: 0.0028
Total loss from exact to noisy: 0.0000
